In [1]:
##Final 


# Research audit note:
# DtACI implements deterministic Algorithm 2 of Gibbs & Candes (2023 preprint).
# AgACI implements Algorithm 1 endpoints and a Python BOA-style gradient trick.
# Exact parity with R OPERA's BOA numerical details is NOT established.
# Original time-series CSV flattening and model helpers still require verification.
# Infinite DtACI intervals are preserved (no artificial clipping); some width
# metrics may be infinite and should be handled appropriately in metrics helper.

# For four datasets
import os
import sys
import time
import glob
import random
import argparse

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

from Helper_Scripts.calculate_metrics import (
    summarize_all_metrics,
    WINDOW as LOCAL_WINDOW,
    CWC_ETA,
)

from Helper_Scripts.models_TLMR import GRUQuantileReg

# ============================================================
# GLOBAL CONFIG
# ============================================================
DATASET_PATHS = [
   
    "../Datasets/TenneTTSO.csv",
    "../Datasets/TransnetBW.csv"

     
]




OUTPUT_CSV = "./results_all_methods_all_datasets.csv"
TRAJECTORY_DIR = "./trajectories/GRU"
SAVE_TRAJECTORIES = True

SEQ_LEN    = 48
EPOCHS     = 200
LR         = 0.001
SEED       = 42
BATCH_SIZE = 32

ALPHA    = 0.1
TINIT    = 5000
SPLIT    = 0.5
GAMMA    = 0.001

DTACI_GAMMA_CANDIDATES = [0.001, 0.002, 0.004, 0.008, 0.0160]
DTACI_INTERVAL_SIZE    = 500

AGACI_GAMMA_CANDIDATES = [0.001, 0.002, 0.004, 0.008, 0.0160]
AGACI_EPS = 0.001

DOWNSAMPLE_STRIDE = 4  # original preprocessing used every fourth observation

METHODS_TO_RUN = ["ACI","LACI","AgACI", "DtACI"]
# METHODS_TO_RUN = ["LACI"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)


# ============================================================
# MODEL: LSTM quantile regression
# ============================================================

def quantile_loss(y_true, y_pred, q):
    diff = y_true - y_pred
    return torch.mean(torch.maximum(q * diff, (q - 1) * diff))


# ============================================================
# DATA LOADING
# ============================================================
class SeqDataset(Dataset):
    def __init__(self, series, seq_len):
        self.X, self.y = [], []
        for i in range(len(series) - seq_len):
            self.X.append(series[i:i + seq_len])
            self.y.append(series[i + seq_len])
        self.X, self.y = np.array(self.X, dtype=np.float32), np.array(self.y, dtype=np.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.float32)


#For 4 datasets : 

def load_dataset(filepath, seq_len=96):
    df = pd.read_csv(filepath)
    y = df.iloc[:, 1:].values.flatten().astype(float)
    y = y[::DOWNSAMPLE_STRIDE]

    # Fit normalization on the historical raw observations only. The first
    # forecast target is y[TINIT + seq_len], so nothing beyond that point
    # may inform the scaler.
    scaler_fit_end = TINIT + seq_len
    if len(y) <= scaler_fit_end:
        raise ValueError(
            f"{filepath}: {len(y)} points after downsampling; "
            f"need more than TINIT + seq_len = {scaler_fit_end}."
        )
    scaler = StandardScaler()
    scaler.fit(y[:scaler_fit_end].reshape(-1, 1))
    y_scaled = scaler.transform(y.reshape(-1, 1)).flatten()

    full_dataset = SeqDataset(y_scaled, seq_len)

    return full_dataset, scaler, y, y_scaled



# ============================================================
# SHARED HELPERS
# ============================================================
def pad(arr, length):
    arr = np.array(arr, dtype=np.float32)
    if len(arr) < length:
        arr = np.concatenate([arr, np.full(length - len(arr), np.nan)])
    return arr


def scoring_function_v1(error_history, magnitude_history):
    if len(error_history) == 0:
        return 0.0
    n_hist = len(error_history)
    exponents = np.arange(n_hist - 1, -1, -1, dtype=np.float64)
    recency_w = 0.85 ** exponents
    #To prevent divding by zero
    mag_w = np.array(magnitude_history, dtype=np.float64) + 1e-4
    joint_w = recency_w * mag_w
    joint_w = joint_w / joint_w.sum()
    return float(np.dot(joint_w, error_history))


def pinball_loss(beta, theta, alpha=ALPHA):
    diff = beta - theta
    return alpha * diff - min(0.0, diff)


def beta_from_scores(score_t, scores_cal):
    beta_t = 1.0 - (np.sum(scores_cal <= score_t) / (len(scores_cal) + 1))
    return float(np.clip(beta_t, 0.0, 1.0))


def _maybe_train(t, tinit, X_train, y_train, model_lower, model_upper, opt_lower, opt_upper, quantiles):
    if t == tinit:
        current_epochs = EPOCHS
    elif (t - tinit) % 6 == 0:
        current_epochs = 1
    else:
        current_epochs = 0

    if current_epochs > 0:
        X_train_tensor = torch.tensor(X_train.reshape(-1, SEQ_LEN), dtype=torch.float32, device=device)
        y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32, device=device)
        train_ds = TensorDataset(X_train_tensor, y_train_tensor)
        train_loader = DataLoader(train_ds, batch_size=min(BATCH_SIZE, len(train_ds)), shuffle=True)

        model_lower.train()
        for _ in range(current_epochs):
            for X_b, y_b in train_loader:
                loss = quantile_loss(y_b, model_lower(X_b), quantiles[0])
                opt_lower.zero_grad(); loss.backward(); opt_lower.step()

        model_upper.train()
        for _ in range(current_epochs):
            for X_b, y_b in train_loader:
                loss = quantile_loss(y_b, model_upper(X_b), quantiles[1])
                opt_upper.zero_grad(); loss.backward(); opt_upper.step()

    return current_epochs


def _fresh_models():
    model_lower = GRUQuantileReg().to(device)
    model_upper = GRUQuantileReg().to(device)
    opt_lower = torch.optim.Adam(model_lower.parameters(), lr=LR)
    opt_upper = torch.optim.Adam(model_upper.parameters(), lr=LR)
    return model_lower, model_upper, opt_lower, opt_upper







def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def interval_error(y, lower, upper):
    return float(not (lower <= y <= upper))


def conformal_radius(scores, alpha_t):
    """Split conformal order statistic; +/- infinity denotes full/empty set.

    Score can be signed (CQR); hence zero radius is NOT imposed.
    At alpha <= 0, set is the entire real line.
    At alpha >= 1, set is empty (represented by q=-inf).
    """
    scores = np.asarray(scores, dtype=np.float64)
    n = len(scores)
    if n == 0:
        raise ValueError("No calibration scores")
    if alpha_t <= 0.0:
        return float("inf")
    if alpha_t >= 1.0:
        return float("-inf")
    rank = int(np.ceil((n + 1) * (1 - alpha_t)))
    if rank > n:
        return float("inf")
    if rank < 1:
        return float("-inf")
    return float(np.partition(scores, rank - 1)[rank - 1])


def conformal_bounds(q_lo, q_hi, scores, alpha_t):
    q = conformal_radius(scores, alpha_t)
    if q == float("inf"):
        return float("-inf"), float("inf")
    if q == float("-inf"):
        return float("inf"), float("-inf")
    return float(q_lo - q), float(q_hi + q)


def predict_quantiles(model_lower, model_upper, X, device):
    x = torch.as_tensor(X, dtype=torch.float32, device=device)
    with torch.no_grad():
        model_lower.eval()
        model_upper.eval()
        lo = model_lower(x).detach().cpu().numpy().reshape(-1)
        hi = model_upper(x).detach().cpu().numpy().reshape(-1)
    return lo, hi


def time_step_data(ds, t, history, split):
    a = t - history
    n_train = int(round(history * split))
    if n_train < 1 or n_train >= history:
        raise ValueError("Require nonempty train and calibration sets")
    return np.arange(a, a+n_train), np.arange(a+n_train, t)

# ============================================================
# METHOD 2: ACI (Eq. 2/4, Gibbs & Candes 2021)
# ============================================================
def run_aci(full_dataset, alpha=ALPHA, gamma=GAMMA, tinit=TINIT, splitSize=SPLIT):
    set_seed(SEED)
    T = len(full_dataset)
    n_pred = T - tinit
    alphaTrajectory = np.zeros(n_pred, dtype=np.float32)
    adaptErrSeq = np.zeros(n_pred, dtype=np.float32)
    y_u_preds = np.zeros(n_pred, dtype=np.float32)
    y_l_preds = np.zeros(n_pred, dtype=np.float32)
    y_true_vals = np.zeros(n_pred, dtype=np.float32)

    alphat = alpha
    WINDOW_SIZE = tinit
    quantiles = [alpha / 2.0, 1.0 - alpha / 2.0]
    model_lower, model_upper, opt_lower, opt_upper = _fresh_models()

    for t in range(tinit, T):
        start_idx = t - WINDOW_SIZE
        current_window_idx = list(range(start_idx, t))
        n_train = int(round(splitSize * WINDOW_SIZE))
        trainPoints = current_window_idx[:n_train]
        calPoints = current_window_idx[n_train:]

        X_train = full_dataset.X[trainPoints].astype(np.float32)
        y_train = full_dataset.y[trainPoints].astype(np.float32)
        X_cal = full_dataset.X[calPoints].astype(np.float32)
        y_cal = full_dataset.y[calPoints].astype(np.float32)
        y_t = float(full_dataset.y[t])

        _maybe_train(t, tinit, X_train, y_train, model_lower, model_upper, opt_lower, opt_upper, quantiles)

        X_cal_tensor = torch.tensor(X_cal.reshape(-1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            model_lower.eval(); model_upper.eval()
            y_l_cal = model_lower(X_cal_tensor).cpu().numpy().reshape(-1)
            y_u_cal = model_upper(X_cal_tensor).cpu().numpy().reshape(-1)
        scores_cal = np.maximum(y_l_cal - y_cal, y_cal - y_u_cal)

        X_t_tensor = torch.tensor(full_dataset.X[t].reshape(1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            q_lo_t = float(model_lower(X_t_tensor).cpu().numpy()[0, 0])
            q_hi_t = float(model_upper(X_t_tensor).cpu().numpy()[0, 0])

        alphat = np.clip(alphat, 1e-6, 1 - 1e-6)
        level_adapt = np.clip((1.0 - alphat) * (1.0 + 1.0 / len(calPoints)), 0.0, 1.0)
        confQuantAdapt = float(np.quantile(scores_cal, level_adapt))

        y_l_adapt = q_lo_t - confQuantAdapt
        y_u_adapt = q_hi_t + confQuantAdapt
        # No artificial clipping of issued intervals.

        err_t = float(not (y_l_adapt <= y_t <= y_u_adapt))
        alphat = np.clip(alphat + gamma * (alpha - err_t), 1e-6, 1 - 1e-6)

        idx = t - tinit
        alphaTrajectory[idx] = alphat
        adaptErrSeq[idx] = err_t
        y_u_preds[idx] = y_u_adapt
        y_l_preds[idx] = y_l_adapt
        y_true_vals[idx] = y_t

        if t % 200 == 0:
            print(f"[ACI] t={t}, alphat={alphat:.4f}, err_t={err_t:.0f}")

    return alphaTrajectory, adaptErrSeq, y_u_preds, y_l_preds, y_true_vals


# ============================================================
# METHOD 3: LACI (user's local variant)
# ============================================================
def run_laci(full_dataset, alpha=ALPHA, gamma=GAMMA, tinit=TINIT, splitSize=SPLIT):
    set_seed(SEED)
    T = len(full_dataset)
    n_pred = T - tinit
    alphaTrajectory = np.zeros(n_pred, dtype=np.float32)
    adaptErrSeq = np.zeros(n_pred, dtype=np.float32)
    y_u_preds = np.zeros(n_pred, dtype=np.float32)
    y_l_preds = np.zeros(n_pred, dtype=np.float32)
    y_true_vals = np.zeros(n_pred, dtype=np.float32)

    alphat = alpha
    WINDOW_SIZE = tinit
    quantiles = [alpha / 2.0, 1.0 - alpha / 2.0]
    error_history, magnitude_history = [], []
    # Coverage of the intervals issued online, rather than retrospective
    # predictions reconstructed using the latest trained model.
    covered_history = []
    local_w = 24
    model_lower, model_upper, opt_lower, opt_upper = _fresh_models()

    for t in range(tinit, T):
        start_idx = t - WINDOW_SIZE
        current_window_idx = list(range(start_idx, t))
        n_train = int(round(splitSize * WINDOW_SIZE))
        trainPoints = current_window_idx[:n_train]
        calPoints = current_window_idx[n_train:]

        X_train = full_dataset.X[trainPoints].astype(np.float32)
        y_train = full_dataset.y[trainPoints].astype(np.float32)
        X_cal = full_dataset.X[calPoints].astype(np.float32)
        y_cal = full_dataset.y[calPoints].astype(np.float32)
        y_t = float(full_dataset.y[t])

        _maybe_train(t, tinit, X_train, y_train, model_lower, model_upper, opt_lower, opt_upper, quantiles)

        X_cal_tensor = torch.tensor(X_cal.reshape(-1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            model_lower.eval(); model_upper.eval()
            y_l_cal = model_lower(X_cal_tensor).cpu().numpy().reshape(-1)
            y_u_cal = model_upper(X_cal_tensor).cpu().numpy().reshape(-1)
        scores_cal = np.maximum(y_l_cal - y_cal, y_cal - y_u_cal)

        alphat = np.clip(alphat, 1e-6, 1 - 1e-6)
        level_adapt = np.clip((1.0 - alphat) * (1.0 + 1.0 / len(calPoints)), 0.0, 1.0)
        confQuantAdapt = float(np.quantile(scores_cal, level_adapt))

        X_t_tensor = torch.tensor(full_dataset.X[t].reshape(1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            q_lo_t = float(model_lower(X_t_tensor).cpu().numpy()[0, 0])
            q_hi_t = float(model_upper(X_t_tensor).cpu().numpy()[0, 0])

        y_l_adapt = q_lo_t - confQuantAdapt
        y_u_adapt = q_hi_t + confQuantAdapt
        # No artificial clipping of issued intervals.

        # Realized local coverage: use only coverage outcomes from intervals
        # actually issued at each past prediction step (including this one).
        binary_err = float(not (y_l_adapt <= y_t <= y_u_adapt))
        covered_history.append(1.0 - binary_err)

        if len(covered_history) >= local_w:
            lpicp = float(np.mean(covered_history[-local_w:]))
            magnitude = float(max((1.0 - alpha) - lpicp, 0.0))
        else:
            # The first w-1 steps have no full online local window.
            # Define their local error as zero, while recording their
            # actual coverage errors for the weighted update.
            magnitude = 0.0

        error_history.append(binary_err)
        magnitude_history.append(magnitude)
        # Keep the original 0.85 decay and +1e-3 epsilon unchanged.
        weighted_err = scoring_function_v1(error_history, magnitude_history)

        alphat = np.clip(alphat + gamma * (alpha - weighted_err), 1e-6, 1 - 1e-6)

        idx = t - tinit
        alphaTrajectory[idx] = alphat
        adaptErrSeq[idx] = binary_err
        y_u_preds[idx] = y_u_adapt
        y_l_preds[idx] = y_l_adapt
        y_true_vals[idx] = y_t

        if t % 200 == 0:
            print(f"[LACI] t={t}, alphat={alphat:.4f}, weighted_err={weighted_err:.4f}")

    return alphaTrajectory, adaptErrSeq, y_u_preds, y_l_preds, y_true_vals


# ============================================================
# METHOD 4: DtACI
# ============================================================
def run_dtaci(full_dataset, alpha=ALPHA, tinit=TINIT, splitSize=SPLIT,
              gamma_candidates=None, interval_size=DTACI_INTERVAL_SIZE):
    """Gibbs & Candes 2023, Algorithm 2 (deterministic DtACI).

    Expert states remain UNPROJECTED. Quantile boundary conventions:
    alpha <= 0 -> R, alpha >= 1 -> empty set.
    Separate LSTMs are trained for this method; no other method's model is used.
    """
    if gamma_candidates is None:
        gamma_candidates = DTACI_GAMMA_CANDIDATES
    gammas = np.asarray(gamma_candidates, dtype=np.float64)
    k = len(gammas)
    if k < 1 or interval_size <= 0:
        raise ValueError("Invalid DtACI gamma candidates or interval size")
    set_seed(SEED)
    T = len(full_dataset)
    n_pred = T-tinit
    alpha_traj = np.empty(n_pred, np.float64)
    errors = np.empty(n_pred, np.float64)
    upper = np.empty(n_pred, np.float64)
    lower = np.empty(n_pred, np.float64)
    truth = np.empty(n_pred, np.float64)
    expert_alpha = np.full(k, alpha, np.float64)
    # Normalized weights, equivalent to the paper's unnormalized recursion.
    weights = np.full(k, 1/k, np.float64)
    sigma = 1.0 / (2.0 * interval_size)
    # Section 3.1 fixed heuristic: E[loss^2] approx (1-alpha)^2 alpha^2 / 3.
    eta = np.sqrt(3 * (np.log(k*interval_size)+2) /
                  ((1-alpha)**2 * alpha**2)) / interval_size
    quantiles = [alpha/2, 1-alpha/2]
    mlo, mhi, olo, ohi = _fresh_models()
    for t in range(tinit, T):
        tr, cal = time_step_data(full_dataset, t, tinit, splitSize)
        _maybe_train(t, tinit, full_dataset.X[tr], full_dataset.y[tr],
                     mlo, mhi, olo, ohi, quantiles)
        cal_l, cal_u = predict_quantiles(mlo, mhi, full_dataset.X[cal], device)
        scores = np.maximum(cal_l-full_dataset.y[cal], full_dataset.y[cal]-cal_u)
        ql, qu = predict_quantiles(mlo, mhi, full_dataset.X[t:t+1], device)
        ql, qu = float(ql[0]), float(qu[0])
        yt = float(full_dataset.y[t])
        beta = beta_from_scores(max(ql-yt, yt-qu), scores)

        probs = weights / weights.sum()
        alpha_out = float(np.dot(probs, expert_alpha))
        lo_t, hi_t = conformal_bounds(ql, qu, scores, alpha_out)
        err_t = interval_error(yt, lo_t, hi_t)

        # Expert losses for past expert states; update AFTER forecasting.
        expert_errors = np.array([
            interval_error(yt, *conformal_bounds(ql, qu, scores, a))
            for a in expert_alpha
        ], dtype=np.float64)
        losses = np.array([pinball_loss(beta, a, alpha) for a in expert_alpha])
        logw = np.log(weights) - eta*losses
        logw -= np.max(logw)
        wbar = np.exp(logw)
        wbar /= wbar.sum()
        weights = (1 - sigma)*wbar + sigma/k
        expert_alpha = np.clip(
    expert_alpha + gammas * (alpha - expert_errors),
    1e-6,
    1.0 - 1e-6
)

        j = t-tinit
        alpha_traj[j], errors[j] = alpha_out, err_t
        lower[j], upper[j], truth[j] = lo_t, hi_t, yt
        if t % 200 == 0:
            print(f"[DtACI] t={t} alpha={alpha_out:.4f} err={err_t:.0f}")
    return alpha_traj, errors, upper, lower, truth


# ============================================================
# METHOD 5: AgACI
# ============================================================
class BOAEndpoint:
    """Adaptive Bernstein expert aggregation with pinball gradient trick.

    This is a transparent Python BOA-style implementation, not a verified
    numerical reproduction of the R OPERA package used by Zaffran et al.
    Separate instances are maintained for each endpoint.
    """
    def __init__(self, n_experts, tau, eps):
        self.k = n_experts
        self.tau = tau
        self.eps = eps
        self.p = np.full(n_experts, 1.0/n_experts)
        self.eta = np.zeros(n_experts)
        self.L = np.zeros(n_experts)
        self.squared = np.zeros(n_experts)
        self.max_abs = np.zeros(n_experts)

    def forecast(self, expert_values):
        return float(self.p @ expert_values)

    def update(self, expert_values, aggregated, observed):
        # d/d forecast of pinball loss rho_tau(y-forecast).
        gradient = float(observed < aggregated) - self.tau
        regrets = gradient * (expert_values - aggregated)
        self.squared += regrets**2
        self.max_abs = np.maximum(self.max_abs, np.abs(regrets))
        E = 2.0**(np.ceil(np.log2(self.max_abs + self.eps)) + 1.0)
        self.L += 0.5*(regrets*(1 + self.eta*regrets) +
                       E*(self.eta*regrets > 0.5))
        candidate = np.full(self.k, np.inf)
        active = self.squared > 0
        candidate[active] = np.sqrt(np.log(self.k)/self.squared[active])
        self.eta = np.minimum(1.0/E, candidate)
        logw = np.log(self.eta) - self.eta*self.L
        logw -= np.max(logw)
        w = np.exp(logw)
        self.p = w/w.sum()


def run_agaci(full_dataset, alpha=ALPHA, tinit=TINIT, splitSize=SPLIT,
              gamma_candidates=None, eps=AGACI_EPS):
    """Zaffran et al. ICML 2022 AgACI: separate endpoint BOA aggregation.

    At each t, every gamma expert creates an ACI interval; the lower and
    upper endpoint forecasts are aggregated with *separate* BOA weights.
    Infinite expert bounds are replaced by predefined finite training-only
    thresholds, as permitted by Algorithm 1's M^(l), M^(u).
    """
    if gamma_candidates is None:
        gamma_candidates = AGACI_GAMMA_CANDIDATES
    gammas = np.asarray(gamma_candidates, dtype=np.float64)
    k = len(gammas)
    if k < 2:
        raise ValueError("BOA aggregation needs at least two experts")
    set_seed(SEED)
    T = len(full_dataset)
    n_pred = T-tinit
    alpha_traj = np.full(n_pred, np.nan, np.float64)  # no single AgACI alpha!
    errors = np.empty(n_pred, np.float64)
    upper = np.empty(n_pred, np.float64)
    lower = np.empty(n_pred, np.float64)
    truth = np.empty(n_pred, np.float64)
    expert_alpha = np.full(k, alpha, np.float64)
    lower_boa = BOAEndpoint(k, alpha/2, eps)
    upper_boa = BOAEndpoint(k, 1-alpha/2, eps)
    quantiles = [alpha/2, 1-alpha/2]
    mlo, mhi, olo, ohi = _fresh_models()

    # Predeclare finite endpoints from TRAINING-ONLY raw-scale standardized
    # history. These are a stated experimental choice for M^(l), M^(u).
    initial = np.asarray(full_dataset.y[:tinit], dtype=float)
    yrange = float(np.ptp(initial))
    margin = 5.0 * max(yrange, 1e-8)
    m_lower = float(np.min(initial) - margin)
    m_upper = float(np.max(initial) + margin)

    for t in range(tinit, T):
        tr, cal = time_step_data(full_dataset, t, tinit, splitSize)
        _maybe_train(t, tinit, full_dataset.X[tr], full_dataset.y[tr],
                     mlo, mhi, olo, ohi, quantiles)
        cal_l, cal_u = predict_quantiles(mlo, mhi, full_dataset.X[cal], device)
        scores = np.maximum(cal_l-full_dataset.y[cal], full_dataset.y[cal]-cal_u)
        ql, qu = predict_quantiles(mlo, mhi, full_dataset.X[t:t+1], device)
        ql, qu = float(ql[0]), float(qu[0])
        yt = float(full_dataset.y[t])

        raw_intervals = [conformal_bounds(ql, qu, scores, a)
                         for a in expert_alpha]
        expert_errors = np.array([
            interval_error(yt, lo, hi) for lo, hi in raw_intervals
        ])
        # BOA requires bounded expert predictions. These thresholds apply
        # only to AgACI's endpoint experts, not to DtACI/LACI predictions.
        expert_lower = np.array([
            np.clip(lo, m_lower, m_upper) for lo, hi in raw_intervals
        ], dtype=float)
        expert_upper = np.array([
            np.clip(hi, m_lower, m_upper) for lo, hi in raw_intervals
        ], dtype=float)
        pred_lower = lower_boa.forecast(expert_lower)
        pred_upper = upper_boa.forecast(expert_upper)
        actual_err = interval_error(yt, pred_lower, pred_upper)

        # Observe target, update each BOA endpoint separately.
        lower_boa.update(expert_lower, pred_lower, yt)
        upper_boa.update(expert_upper, pred_upper, yt)
        # Each expert follows its own ACI feedback, not the aggregate error.
        expert_alpha = np.clip(
    expert_alpha + gammas * (alpha - expert_errors),
    1e-6,
    1.0 - 1e-6
)

        j = t-tinit
        errors[j] = actual_err
        lower[j], upper[j], truth[j] = pred_lower, pred_upper, yt
        if t % 200 == 0:
            print(f"[AgACI] t={t} err={actual_err:.0f} "
                  f"interval=[{pred_lower:.3f}, {pred_upper:.3f}]")
    return alpha_traj, errors, upper, lower, truth


METHOD_FUNCS = {
   
    "ACI": run_aci,
    "LACI": run_laci,
    "DtACI": run_dtaci,
    "AgACI": run_agaci,
}


# ============================================================
# DRIVER
# ============================================================
def run_one_method(dataset_name, method_name, full_dataset, scaler, **kwargs):
    print(f"\n{'=' * 70}\n{dataset_name} :: {method_name}\n{'=' * 70}")
    run_fn = METHOD_FUNCS[method_name]
    start_time = time.time()
    alphaTraj, errSeq, y_u_preds, y_l_preds, y_true_vals = run_fn(full_dataset, **kwargs)
    time_taken = time.time() - start_time

    # Direct inverse affine mapping also preserves +/- infinity, which is
    # an intentional DtACI prediction-set boundary convention.
    offset = float(scaler.mean_[0])
    scale = float(scaler.scale_[0])
    y_u_inv = y_u_preds * scale + offset
    y_l_inv = y_l_preds * scale + offset
    y_true_inv = y_true_vals * scale + offset

    if np.isfinite(y_u_inv).all() and np.isfinite(y_l_inv).all():
        metrics = summarize_all_metrics(
            y_true_inv, y_l_inv, y_u_inv,
            alpha=ALPHA, w=LOCAL_WINDOW, eta=CWC_ETA,
            time_taken=time_taken,
        )
    else:
        # Do not substitute finite prediction bounds for the actual infinite
        # DtACI intervals. Report coverage and strictly defined width stats.
        covered = (y_true_inv >= y_l_inv) & (y_true_inv <= y_u_inv)
        width = np.maximum(y_u_inv - y_l_inv, 0.0)
        local_coverage = np.array([
            covered[i:i+LOCAL_WINDOW].mean()
            for i in range(max(0, len(covered)-LOCAL_WINDOW+1))
        ])
        metrics = {
            'PICP': float(covered.mean()),
            'MPIW': float(np.mean(width)),
            'MLPICP': float(np.mean(local_coverage)) if len(local_coverage) else np.nan,
            'LPICE': np.nan, 'LPIA': np.nan,
            'LMPIW': np.nan, 'LCWC': np.nan, 'STD_LE': np.nan,
            'n_local_windows': int(len(local_coverage)),
            'time_sec': float(time_taken),
            'fraction_infinite': float(np.mean(~np.isfinite(y_l_inv) | ~np.isfinite(y_u_inv))),
        }
        print('[INFO] Unbounded intervals: width-dependent custom metrics are NaN, not imputed.')

    print(f"[{dataset_name}/{method_name}] " +
          " | ".join(f"{k}={v:.4f}" for k, v in metrics.items()
                      if isinstance(v, float) and k != "time_sec"))
    print(f"[{dataset_name}/{method_name}] time_sec={time_taken:.2f}")

    if SAVE_TRAJECTORIES:
        os.makedirs(TRAJECTORY_DIR, exist_ok=True)
        max_len = max(len(alphaTraj), len(errSeq), len(y_u_inv), len(y_l_inv), len(y_true_inv))
        traj_df = pd.DataFrame({
            "step": np.arange(max_len),
            "alphaTrajectory": pad(alphaTraj, max_len),
            "errSeq": pad(errSeq, max_len),
            "y_u_pred": pad(y_u_inv, max_len),
            "y_l_pred": pad(y_l_inv, max_len),
            "y_true": pad(y_true_inv, max_len),
        })
        safe_ds = os.path.splitext(os.path.basename(dataset_name))[0]
        traj_df.to_csv(os.path.join(TRAJECTORY_DIR, f"{safe_ds}__{method_name}.csv"), index=False)

    return metrics


def main(dataset_paths):
    if not dataset_paths:
        print("No datasets provided. Edit DATASET_PATHS at the top of this "
              "script, or pass paths with --data path1.csv path2.csv ...")
        sys.exit(1)

    all_rows = []
    for dataset_path in dataset_paths:
        print(f"\n\n########## Loading dataset: {dataset_path} ##########")
        full_dataset, scaler, y_raw, y_scaled = load_dataset(dataset_path, seq_len=SEQ_LEN)
        print(f"Series length: {len(y_raw)}, usable (t - seq_len) pairs: {len(full_dataset)}")

        if len(full_dataset) <= TINIT:
            print(f"WARNING: dataset {dataset_path} has fewer than TINIT={TINIT} "
                  f"usable points ({len(full_dataset)}). Skipping.")
            continue

        for method_name in METHODS_TO_RUN:
            kwargs = dict(alpha=ALPHA, tinit=TINIT, splitSize=SPLIT)
            if method_name == "Normal_QR":
                kwargs = dict(alpha=ALPHA, tinit=TINIT, splitSize=1)
            elif method_name in ("ACI", "LACI"):
                kwargs["gamma"] = GAMMA
            elif method_name == "DtACI":
                kwargs["gamma_candidates"] = DTACI_GAMMA_CANDIDATES
                kwargs["interval_size"] = DTACI_INTERVAL_SIZE
            elif method_name == "AgACI":
                kwargs["gamma_candidates"] = AGACI_GAMMA_CANDIDATES
                kwargs["eps"] = AGACI_EPS

            metrics = run_one_method(dataset_path, method_name, full_dataset, scaler, **kwargs)
            row = {"dataset": os.path.basename(dataset_path), "method": method_name}
            row.update(metrics)
            all_rows.append(row)

            pd.DataFrame(all_rows).to_csv(OUTPUT_CSV, index=False)

    results_df = pd.DataFrame(all_rows)
    col_order = ["dataset", "method", "PICP", "MPIW", "MLPICP", "LPICE", "LPIA",
                 "LMPIW", "LCWC", "STD_LE", "n_local_windows", "time_sec"]
    col_order = [c for c in col_order if c in results_df.columns]
    results_df = results_df[col_order + [c for c in results_df.columns if c not in col_order]]
    results_df.to_csv(OUTPUT_CSV, index=False)

    print(f"\nAll results written to: {OUTPUT_CSV}")
    if SAVE_TRAJECTORIES:
        print(f"Per-step trajectories written to: {TRAJECTORY_DIR}/")
    print("\n", results_df.to_string(index=False))
    return results_df


if __name__ == "__main__":
    dataset_paths = DATASET_PATHS
    main(dataset_paths)

device: cuda


########## Loading dataset: ../Datasets/TenneTTSO.csv ##########
Series length: 9528, usable (t - seq_len) pairs: 9480

../Datasets/TenneTTSO.csv :: ACI
[ACI] t=5000, alphat=0.1001, err_t=0
[ACI] t=5200, alphat=0.1001, err_t=0
[ACI] t=5400, alphat=0.1031, err_t=0
[ACI] t=5600, alphat=0.1141, err_t=0
[ACI] t=5800, alphat=0.1171, err_t=0
[ACI] t=6000, alphat=0.1261, err_t=0
[ACI] t=6200, alphat=0.1371, err_t=0
[ACI] t=6400, alphat=0.1401, err_t=0
[ACI] t=6600, alphat=0.1431, err_t=0
[ACI] t=6800, alphat=0.1451, err_t=0
[ACI] t=7000, alphat=0.1401, err_t=0
[ACI] t=7200, alphat=0.1471, err_t=0
[ACI] t=7400, alphat=0.1431, err_t=0
[ACI] t=7600, alphat=0.1151, err_t=1
[ACI] t=7800, alphat=0.1241, err_t=0
[ACI] t=8000, alphat=0.1371, err_t=0
[ACI] t=8200, alphat=0.1221, err_t=0
[ACI] t=8400, alphat=0.1361, err_t=0
[ACI] t=8600, alphat=0.1331, err_t=0
[ACI] t=8800, alphat=0.1191, err_t=1
[ACI] t=9000, alphat=0.1211, err_t=0
[ACI] t=9200, alphat=0.1231, err_t=0
[ACI] t=9400, alph

In [ ]:
# For location datasets : 


"""Location 1/2: ACI, LACI, DtACI and AgACI online comparison.

Requires your existing Helper_Scripts and ../Datasets/Location1.csv and
Location2.csv, each containing a numeric Power column.
""" 



# For Location 1 and Location 2
import os
import sys
import time
import glob
import random
import argparse

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

from Helper_Scripts.calculate_metrics import (
    summarize_all_metrics,
    WINDOW as LOCAL_WINDOW,
    CWC_ETA,
)

from Helper_Scripts.models_TLMR import GRUQuantileReg

# ============================================================
# GLOBAL CONFIG
# ============================================================
DATASET_PATHS = [
    "../Datasets/Location1.csv",
    "../Datasets/Location2.csv",
]

OUTPUT_CSV = "./results_locations_1_2.csv"
TRAJECTORY_DIR = "./trajectories/GRU"
SAVE_TRAJECTORIES = True

SEQ_LEN    = 96
EPOCHS     = 200
LR         = 0.001
SEED       = 42
BATCH_SIZE = 32

ALPHA    = 0.1
TINIT    = 5000
SPLIT    = 0.5
GAMMA    = 0.001

DTACI_GAMMA_CANDIDATES = [0.001, 0.002, 0.004, 0.008, 0.016, 0.032, 0.064, 0.128]
DTACI_INTERVAL_SIZE    = 500

AGACI_GAMMA_CANDIDATES = [0.001, 0.002, 0.004, 0.008, 0.0160]
AGACI_EPS = 0.001



METHODS_TO_RUN = ["ACI", "LACI", "DtACI", "AgACI"]


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)


# ============================================================
# MODEL: LSTM quantile regression
# ============================================================

def quantile_loss(y_true, y_pred, q):
    diff = y_true - y_pred
    return torch.mean(torch.maximum(q * diff, (q - 1) * diff))


# ============================================================
# DATA LOADING
# ============================================================
class SeqDataset(Dataset):
    def __init__(self, series, seq_len):
        self.X, self.y = [], []
        for i in range(len(series) - seq_len):
            self.X.append(series[i:i + seq_len])
            self.y.append(series[i + seq_len])
        self.X, self.y = np.array(self.X, dtype=np.float32), np.array(self.y, dtype=np.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.float32)


#For Location 1 and Location 2: 

def load_dataset(filepath, seq_len=SEQ_LEN):
    """Preserve order, use Power only, fit scaler before first test target."""
    df = pd.read_csv(filepath).copy()
    if "Power" not in df.columns:
        raise ValueError(f"{filepath}: expected numeric Power column")
    y = pd.to_numeric(df["Power"], errors="raise").to_numpy(dtype=np.float64)
    y = y[::4]
    if not np.isfinite(y).all():
        raise ValueError(f"{filepath}: Power contains NaN/Inf; fix data explicitly")
    fit_end = TINIT + seq_len  # first forecast target is y[fit_end]
    if len(y) <= fit_end:
        raise ValueError(f"{filepath}: need >{fit_end} observations, got {len(y)}")
    scaler = StandardScaler()
    scaler.fit(y[:fit_end].reshape(-1, 1))
    y_scaled = scaler.transform(y.reshape(-1, 1)).ravel()
    return SeqDataset(y_scaled, seq_len), scaler, y, y_scaled


# ============================================================
# SHARED HELPERS
# ============================================================
def pad(arr, length):
    arr = np.array(arr, dtype=np.float64)
    if len(arr) < length:
        arr = np.concatenate([arr, np.full(length - len(arr), np.nan)])
    return arr


def scoring_function_v1(error_history, magnitude_history):
    if len(error_history) == 0:
        return 0.0
    n_hist = len(error_history)
    exponents = np.arange(n_hist - 1, -1, -1, dtype=np.float64)
    recency_w = 0.85 ** exponents
    #To prevent divding by zero
    mag_w = np.array(magnitude_history, dtype=np.float64) + 1e-4
    joint_w = recency_w * mag_w
    joint_w = joint_w / joint_w.sum()
    return float(np.dot(joint_w, error_history))


def pinball_loss(beta, theta, alpha=ALPHA):
    diff = beta - theta
    return alpha * diff - min(0.0, diff)


def beta_from_scores(score_t, scores_cal):
    beta_t = 1.0 - (np.sum(scores_cal <= score_t) / (len(scores_cal) + 1))
    return float(np.clip(beta_t, 0.0, 1.0))


def _maybe_train(t, tinit, X_train, y_train, model_lower, model_upper, opt_lower, opt_upper, quantiles):
    if t == tinit:
        current_epochs = EPOCHS
    elif (t - tinit) % 6 == 0:
        current_epochs = 1
    else:
        current_epochs = 0

    if current_epochs > 0:
        X_train_tensor = torch.tensor(X_train.reshape(-1, SEQ_LEN), dtype=torch.float32, device=device)
        y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32, device=device)
        train_ds = TensorDataset(X_train_tensor, y_train_tensor)
        train_loader = DataLoader(train_ds, batch_size=min(BATCH_SIZE, len(train_ds)), shuffle=True)

        model_lower.train()
        for _ in range(current_epochs):
            for X_b, y_b in train_loader:
                loss = quantile_loss(y_b, model_lower(X_b), quantiles[0])
                opt_lower.zero_grad(); loss.backward(); opt_lower.step()

        model_upper.train()
        for _ in range(current_epochs):
            for X_b, y_b in train_loader:
                loss = quantile_loss(y_b, model_upper(X_b), quantiles[1])
                opt_upper.zero_grad(); loss.backward(); opt_upper.step()

    return current_epochs


def _fresh_models():
    model_lower = GRUQuantileReg().to(device)
    model_upper = GRUQuantileReg().to(device)
    opt_lower = torch.optim.Adam(model_lower.parameters(), lr=LR)
    opt_upper = torch.optim.Adam(model_upper.parameters(), lr=LR)
    return model_lower, model_upper, opt_lower, opt_upper







def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def interval_error(y, lower, upper):
    return float(not (lower <= y <= upper))


def conformal_radius(scores, alpha_t):
    """Split conformal order statistic; +/- infinity denotes full/empty set.

    Score can be signed (CQR); hence zero radius is NOT imposed.
    At alpha <= 0, set is the entire real line.
    At alpha >= 1, set is empty (represented by q=-inf).
    """
    scores = np.asarray(scores, dtype=np.float64)
    n = len(scores)
    if n == 0:
        raise ValueError("No calibration scores")
    if alpha_t <= 0.0:
        return float("inf")
    if alpha_t >= 1.0:
        return float("-inf")
    rank = int(np.ceil((n + 1) * (1 - alpha_t)))
    if rank > n:
        return float("inf")
    if rank < 1:
        return float("-inf")
    return float(np.partition(scores, rank - 1)[rank - 1])


def conformal_bounds(q_lo, q_hi, scores, alpha_t):
    q = conformal_radius(scores, alpha_t)
    if q == float("inf"):
        return float("-inf"), float("inf")
    if q == float("-inf"):
        return float("inf"), float("-inf")
    return float(q_lo - q), float(q_hi + q)


def predict_quantiles(model_lower, model_upper, X, device):
    x = torch.as_tensor(X, dtype=torch.float32, device=device)
    with torch.no_grad():
        model_lower.eval()
        model_upper.eval()
        lo = model_lower(x).detach().cpu().numpy().reshape(-1)
        hi = model_upper(x).detach().cpu().numpy().reshape(-1)
    return lo, hi


def time_step_data(ds, t, history, split):
    a = t - history
    n_train = int(round(history * split))
    if n_train < 1 or n_train >= history:
        raise ValueError("Require nonempty train and calibration sets")
    return np.arange(a, a+n_train), np.arange(a+n_train, t)

# ============================================================
# METHOD 2: ACI (Eq. 2/4, Gibbs & Candes 2021)
# ============================================================
def run_aci(full_dataset, alpha=ALPHA, gamma=GAMMA, tinit=TINIT, splitSize=SPLIT):
    set_seed(SEED)
    T = len(full_dataset)
    n_pred = T - tinit
    alphaTrajectory = np.zeros(n_pred, dtype=np.float32)
    adaptErrSeq = np.zeros(n_pred, dtype=np.float32)
    y_u_preds = np.zeros(n_pred, dtype=np.float32)
    y_l_preds = np.zeros(n_pred, dtype=np.float32)
    y_true_vals = np.zeros(n_pred, dtype=np.float32)

    alphat = alpha
    WINDOW_SIZE = tinit
    quantiles = [alpha / 2.0, 1.0 - alpha / 2.0]
    model_lower, model_upper, opt_lower, opt_upper = _fresh_models()

    for t in range(tinit, T):
        start_idx = t - WINDOW_SIZE
        current_window_idx = list(range(start_idx, t))
        n_train = int(round(splitSize * WINDOW_SIZE))
        trainPoints = current_window_idx[:n_train]
        calPoints = current_window_idx[n_train:]

        X_train = full_dataset.X[trainPoints].astype(np.float32)
        y_train = full_dataset.y[trainPoints].astype(np.float32)
        X_cal = full_dataset.X[calPoints].astype(np.float32)
        y_cal = full_dataset.y[calPoints].astype(np.float32)
        y_t = float(full_dataset.y[t])

        _maybe_train(t, tinit, X_train, y_train, model_lower, model_upper, opt_lower, opt_upper, quantiles)

        X_cal_tensor = torch.tensor(X_cal.reshape(-1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            model_lower.eval(); model_upper.eval()
            y_l_cal = model_lower(X_cal_tensor).cpu().numpy().reshape(-1)
            y_u_cal = model_upper(X_cal_tensor).cpu().numpy().reshape(-1)
        scores_cal = np.maximum(y_l_cal - y_cal, y_cal - y_u_cal)

        X_t_tensor = torch.tensor(full_dataset.X[t].reshape(1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            q_lo_t = float(model_lower(X_t_tensor).cpu().numpy()[0, 0])
            q_hi_t = float(model_upper(X_t_tensor).cpu().numpy()[0, 0])

        # Match the four-dataset ACI quantile selection (interpolated np.quantile).
        alphat = float(np.clip(alphat, 1e-6, 1.0 - 1e-6))
        level_adapt = float(np.clip(
            (1.0 - alphat) * (1.0 + 1.0 / len(calPoints)), 0.0, 1.0
        ))
        radius = float(np.quantile(scores_cal, level_adapt))
        y_l_adapt, y_u_adapt = q_lo_t - radius, q_hi_t + radius
        # No artificial clipping of prediction interval endpoints.

        err_t = float(not (y_l_adapt <= y_t <= y_u_adapt))
        alphat = float(np.clip(alphat + gamma * (alpha - err_t), 1e-6, 1.0 - 1e-6))

        idx = t - tinit
        alphaTrajectory[idx] = alphat
        adaptErrSeq[idx] = err_t
        y_u_preds[idx] = y_u_adapt
        y_l_preds[idx] = y_l_adapt
        y_true_vals[idx] = y_t

        if t % 200 == 0:
            print(f"[ACI] t={t}, alphat={alphat:.4f}, err_t={err_t:.0f}")

    return alphaTrajectory, adaptErrSeq, y_u_preds, y_l_preds, y_true_vals


# ============================================================
# METHOD 3: LACI (user's local variant)
# ============================================================
def run_laci(full_dataset, alpha=ALPHA, gamma=GAMMA, tinit=TINIT, splitSize=SPLIT):
    set_seed(SEED)
    T = len(full_dataset)
    n_pred = T - tinit
    alphaTrajectory = np.zeros(n_pred, dtype=np.float32)
    adaptErrSeq = np.zeros(n_pred, dtype=np.float32)
    y_u_preds = np.zeros(n_pred, dtype=np.float32)
    y_l_preds = np.zeros(n_pred, dtype=np.float32)
    y_true_vals = np.zeros(n_pred, dtype=np.float32)

    alphat = alpha
    WINDOW_SIZE = tinit
    quantiles = [alpha / 2.0, 1.0 - alpha / 2.0]
    error_history, magnitude_history = [], []
    # Coverage of the intervals issued online, rather than retrospective
    # predictions reconstructed using the latest trained model.
    covered_history = []
    local_w = 24
    model_lower, model_upper, opt_lower, opt_upper = _fresh_models()

    for t in range(tinit, T):
        start_idx = t - WINDOW_SIZE
        current_window_idx = list(range(start_idx, t))
        n_train = int(round(splitSize * WINDOW_SIZE))
        trainPoints = current_window_idx[:n_train]
        calPoints = current_window_idx[n_train:]

        X_train = full_dataset.X[trainPoints].astype(np.float32)
        y_train = full_dataset.y[trainPoints].astype(np.float32)
        X_cal = full_dataset.X[calPoints].astype(np.float32)
        y_cal = full_dataset.y[calPoints].astype(np.float32)
        y_t = float(full_dataset.y[t])

        _maybe_train(t, tinit, X_train, y_train, model_lower, model_upper, opt_lower, opt_upper, quantiles)

        X_cal_tensor = torch.tensor(X_cal.reshape(-1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            model_lower.eval(); model_upper.eval()
            y_l_cal = model_lower(X_cal_tensor).cpu().numpy().reshape(-1)
            y_u_cal = model_upper(X_cal_tensor).cpu().numpy().reshape(-1)
        scores_cal = np.maximum(y_l_cal - y_cal, y_cal - y_u_cal)

        # Compute base quantile forecasts, then select the adaptive calibration quantile.

        X_t_tensor = torch.tensor(full_dataset.X[t].reshape(1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            q_lo_t = float(model_lower(X_t_tensor).cpu().numpy()[0, 0])
            q_hi_t = float(model_upper(X_t_tensor).cpu().numpy()[0, 0])

        # Match the four-dataset LACI quantile selection (interpolated np.quantile).
        alphat = float(np.clip(alphat, 1e-6, 1.0 - 1e-6))
        level_adapt = float(np.clip(
            (1.0 - alphat) * (1.0 + 1.0 / len(calPoints)), 0.0, 1.0
        ))
        radius = float(np.quantile(scores_cal, level_adapt))
        y_l_adapt, y_u_adapt = q_lo_t - radius, q_hi_t + radius
        # No artificial clipping of prediction interval endpoints.

        # Realized local coverage: use only coverage outcomes from intervals
        # actually issued at each past prediction step (including this one).
        binary_err = float(not (y_l_adapt <= y_t <= y_u_adapt))
        covered_history.append(1.0 - binary_err)

        if len(covered_history) >= local_w:
            lpicp = float(np.mean(covered_history[-local_w:]))
            magnitude = float(max((1.0 - alpha) - lpicp, 0.0))
        else:
            # The first w-1 steps have no full online local window.
            # Define their local error as zero, while recording their
            # actual coverage errors for the weighted update.
            magnitude = 0.0

        error_history.append(binary_err)
        magnitude_history.append(magnitude)
        # scoring_function_v1 uses decay 0.95 and epsilon 1e-5.
        weighted_err = scoring_function_v1(error_history, magnitude_history)

        alphat = float(np.clip(alphat + gamma * (alpha - weighted_err), 1e-6, 1.0 - 1e-6))

        idx = t - tinit
        alphaTrajectory[idx] = alphat
        adaptErrSeq[idx] = binary_err
        y_u_preds[idx] = y_u_adapt
        y_l_preds[idx] = y_l_adapt
        y_true_vals[idx] = y_t

        if t % 200 == 0:
            print(f"[LACI] t={t}, alphat={alphat:.4f}, weighted_err={weighted_err:.4f}")

    return alphaTrajectory, adaptErrSeq, y_u_preds, y_l_preds, y_true_vals


# ============================================================
# METHOD 4: DtACI
# ============================================================
def run_dtaci(full_dataset, alpha=ALPHA, tinit=TINIT, splitSize=SPLIT,
              gamma_candidates=None, interval_size=DTACI_INTERVAL_SIZE):
    """Clipped deterministic DtACI, based on Gibbs & Candes Algorithm 2.

    Expert alpha states are projected into [1e-6, 1-1e-6]. Conventions:
    alpha <= 0 -> R, alpha >= 1 -> empty set.
    Separate LSTMs are trained for this method; no other method's model is used.
    """
    if gamma_candidates is None:
        gamma_candidates = DTACI_GAMMA_CANDIDATES
    gammas = np.asarray(gamma_candidates, dtype=np.float64)
    k = len(gammas)
    if k < 1 or interval_size <= 0:
        raise ValueError("Invalid DtACI gamma candidates or interval size")
    set_seed(SEED)
    T = len(full_dataset)
    n_pred = T-tinit
    alpha_traj = np.empty(n_pred, np.float64)
    errors = np.empty(n_pred, np.float64)
    upper = np.empty(n_pred, np.float64)
    lower = np.empty(n_pred, np.float64)
    truth = np.empty(n_pred, np.float64)
    expert_alpha = np.full(k, alpha, np.float64)
    # Normalized weights, equivalent to the paper's unnormalized recursion.
    weights = np.full(k, 1/k, np.float64)
    sigma = 1.0 / (2.0 * interval_size)
    # Section 3.1 fixed heuristic: E[loss^2] approx (1-alpha)^2 alpha^2 / 3.
    eta = np.sqrt(3 * (np.log(k*interval_size)+2) /
                  ((1-alpha)**2 * alpha**2)) / interval_size
    quantiles = [alpha/2, 1-alpha/2]
    mlo, mhi, olo, ohi = _fresh_models()
    for t in range(tinit, T):
        tr, cal = time_step_data(full_dataset, t, tinit, splitSize)
        _maybe_train(t, tinit, full_dataset.X[tr], full_dataset.y[tr],
                     mlo, mhi, olo, ohi, quantiles)
        cal_l, cal_u = predict_quantiles(mlo, mhi, full_dataset.X[cal], device)
        scores = np.maximum(cal_l-full_dataset.y[cal], full_dataset.y[cal]-cal_u)
        ql, qu = predict_quantiles(mlo, mhi, full_dataset.X[t:t+1], device)
        ql, qu = float(ql[0]), float(qu[0])
        yt = float(full_dataset.y[t])
        beta = beta_from_scores(max(ql-yt, yt-qu), scores)

        probs = weights / weights.sum()
        alpha_out = float(np.dot(probs, expert_alpha))
        lo_t, hi_t = conformal_bounds(ql, qu, scores, alpha_out)
        err_t = interval_error(yt, lo_t, hi_t)

        # Expert losses for past expert states; update AFTER forecasting.
        expert_errors = np.array([
            interval_error(yt, *conformal_bounds(ql, qu, scores, a))
            for a in expert_alpha
        ], dtype=np.float64)
        losses = np.array([pinball_loss(beta, a, alpha) for a in expert_alpha])
        logw = np.log(weights) - eta*losses
        logw -= np.max(logw)
        wbar = np.exp(logw)
        wbar /= wbar.sum()
        weights = (1 - sigma)*wbar + sigma/k
        expert_alpha = np.clip(
            expert_alpha + gammas * (alpha - expert_errors),
            1e-6, 1.0 - 1e-6
        )

        j = t-tinit
        alpha_traj[j], errors[j] = alpha_out, err_t
        lower[j], upper[j], truth[j] = lo_t, hi_t, yt
        if t % 200 == 0:
            print(f"[DtACI] t={t} alpha={alpha_out:.4f} err={err_t:.0f}")
    return alpha_traj, errors, upper, lower, truth


# ============================================================
# METHOD 5: AgACI
# ============================================================
class BOAEndpoint:
    """Adaptive Bernstein expert aggregation with pinball gradient trick.

    This is a transparent Python BOA-style implementation, not a verified
    numerical reproduction of the R OPERA package used by Zaffran et al.
    Separate instances are maintained for each endpoint.
    """
    def __init__(self, n_experts, tau, eps):
        self.k = n_experts
        self.tau = tau
        self.eps = eps
        self.p = np.full(n_experts, 1.0/n_experts)
        self.eta = np.zeros(n_experts)
        self.L = np.zeros(n_experts)
        self.squared = np.zeros(n_experts)
        self.max_abs = np.zeros(n_experts)

    def forecast(self, expert_values):
        return float(self.p @ expert_values)

    def update(self, expert_values, aggregated, observed):
        # d/d forecast of pinball loss rho_tau(y-forecast).
        gradient = float(observed < aggregated) - self.tau
        regrets = gradient * (expert_values - aggregated)
        self.squared += regrets**2
        self.max_abs = np.maximum(self.max_abs, np.abs(regrets))
        E = 2.0**(np.ceil(np.log2(self.max_abs + self.eps)) + 1.0)
        self.L += 0.5*(regrets*(1 + self.eta*regrets) +
                       E*(self.eta*regrets > 0.5))
        candidate = np.full(self.k, np.inf)
        active = self.squared > 0
        candidate[active] = np.sqrt(np.log(self.k)/self.squared[active])
        self.eta = np.minimum(1.0/E, candidate)
        logw = np.log(self.eta) - self.eta*self.L
        logw -= np.max(logw)
        w = np.exp(logw)
        self.p = w/w.sum()


def run_agaci(full_dataset, alpha=ALPHA, tinit=TINIT, splitSize=SPLIT,
              gamma_candidates=None, eps=AGACI_EPS):
    """Clipped AgACI based on Zaffran et al.: separate endpoint BOA aggregation.

    At each t, every gamma expert creates an ACI interval; the lower and
    upper endpoint forecasts are aggregated with *separate* BOA weights.
    Infinite expert bounds are replaced by predefined finite training-only
    thresholds, as permitted by Algorithm 1's M^(l), M^(u).
    """
    if gamma_candidates is None:
        gamma_candidates = AGACI_GAMMA_CANDIDATES
    gammas = np.asarray(gamma_candidates, dtype=np.float64)
    k = len(gammas)
    if k < 2:
        raise ValueError("BOA aggregation needs at least two experts")
    set_seed(SEED)
    T = len(full_dataset)
    n_pred = T-tinit
    alpha_traj = np.full(n_pred, np.nan, np.float64)  # no single AgACI alpha!
    errors = np.empty(n_pred, np.float64)
    upper = np.empty(n_pred, np.float64)
    lower = np.empty(n_pred, np.float64)
    truth = np.empty(n_pred, np.float64)
    expert_alpha = np.full(k, alpha, np.float64)
    lower_boa = BOAEndpoint(k, alpha/2, eps)
    upper_boa = BOAEndpoint(k, 1-alpha/2, eps)
    quantiles = [alpha/2, 1-alpha/2]
    mlo, mhi, olo, ohi = _fresh_models()

    # Predeclare finite endpoints from TRAINING-ONLY raw-scale standardized
    # history. These are a stated experimental choice for M^(l), M^(u).
    initial = np.asarray(full_dataset.y[:tinit], dtype=float)
    yrange = float(np.ptp(initial))
    margin = 5.0 * max(yrange, 1e-8)
    m_lower = float(np.min(initial) - margin)
    m_upper = float(np.max(initial) + margin)

    for t in range(tinit, T):
        tr, cal = time_step_data(full_dataset, t, tinit, splitSize)
        _maybe_train(t, tinit, full_dataset.X[tr], full_dataset.y[tr],
                     mlo, mhi, olo, ohi, quantiles)
        cal_l, cal_u = predict_quantiles(mlo, mhi, full_dataset.X[cal], device)
        scores = np.maximum(cal_l-full_dataset.y[cal], full_dataset.y[cal]-cal_u)
        ql, qu = predict_quantiles(mlo, mhi, full_dataset.X[t:t+1], device)
        ql, qu = float(ql[0]), float(qu[0])
        yt = float(full_dataset.y[t])

        raw_intervals = [conformal_bounds(ql, qu, scores, a)
                         for a in expert_alpha]
        expert_errors = np.array([
            interval_error(yt, lo, hi) for lo, hi in raw_intervals
        ])
        # BOA requires bounded expert predictions. These thresholds apply
        # only to AgACI's endpoint experts, not to DtACI/LACI predictions.
        expert_lower = np.array([
            np.clip(lo, m_lower, m_upper) for lo, hi in raw_intervals
        ], dtype=float)
        expert_upper = np.array([
            np.clip(hi, m_lower, m_upper) for lo, hi in raw_intervals
        ], dtype=float)
        pred_lower = lower_boa.forecast(expert_lower)
        pred_upper = upper_boa.forecast(expert_upper)
        actual_err = interval_error(yt, pred_lower, pred_upper)

        # Observe target, update each BOA endpoint separately.
        lower_boa.update(expert_lower, pred_lower, yt)
        upper_boa.update(expert_upper, pred_upper, yt)
        # Each expert follows its own ACI feedback, not the aggregate error.
        expert_alpha = np.clip(
            expert_alpha + gammas * (alpha - expert_errors),
            1e-6, 1.0 - 1e-6
        )

        j = t-tinit
        errors[j] = actual_err
        lower[j], upper[j], truth[j] = pred_lower, pred_upper, yt
        if t % 200 == 0:
            print(f"[AgACI] t={t} err={actual_err:.0f} "
                  f"interval=[{pred_lower:.3f}, {pred_upper:.3f}]")
    return alpha_traj, errors, upper, lower, truth


METHOD_FUNCS = {
   
    "ACI": run_aci,
    "LACI": run_laci,
    "DtACI": run_dtaci,
    "AgACI": run_agaci,
}


# ============================================================
# EVALUATION FALLBACK: KEEP EVERY FORECAST AND EVERY LOCAL WINDOW
# ============================================================
def metrics_all_points_fallback(y_true, lower, upper, elapsed, alpha=ALPHA,
                                window=LOCAL_WINDOW):
    """Coverage metrics with no excluded rows or windows.

    Finite crossed intervals are empty prediction sets and count as misses.
    Full-line intervals (-inf, +inf) have coverage 1 and infinite width.
    Empty intervals (+inf, -inf) have coverage 0 and zero width.
    Width-dependent metrics with infinite widths are retained as inf or NaN.
    """
    y_true = np.asarray(y_true, dtype=np.float64)
    lower = np.asarray(lower, dtype=np.float64)
    upper = np.asarray(upper, dtype=np.float64)
    covered = ((lower <= y_true) & (y_true <= upper)).astype(np.float64)
    width = np.where(lower > upper, 0.0, upper - lower)
    # For atypical nonfinite combinations, do not fabricate a finite width.
    with np.errstate(invalid="ignore"):
        width = np.where(np.isnan(width), np.nan, np.maximum(width, 0.0))

    n_windows = max(0, len(covered) - window + 1)
    local_cov = np.array([
        covered[i:i + window].mean() for i in range(n_windows)
    ], dtype=np.float64)
    local_width = np.array([
        width[i:i + window].mean() for i in range(n_windows)
    ], dtype=np.float64)
    target = 1.0 - alpha
    local_err = np.maximum(target - local_cov, 0.0)
    with np.errstate(invalid="ignore"):
        return {
            "PICP": float(covered.mean()),
            "MPIW": float(width.mean()),
            "MLPICP": float(local_cov.mean()) if n_windows else np.nan,
            "LPICE": float(local_err.mean()) if n_windows else np.nan,
            "LPIA": float(np.mean(local_cov >= target)) if n_windows else np.nan,
            "LMPIW": float(local_width.mean()) if n_windows else np.nan,
            "LCWC": np.nan,  # external helper not defined for exceptional widths
            "STD_LE": float(np.std(local_err)) if n_windows else np.nan,
            "n_local_windows": int(n_windows),
            "time_sec": float(elapsed),
            "n_finite_crossed": int(np.sum(
                (lower > upper) & np.isfinite(lower) & np.isfinite(upper)
            )),
            "n_excluded_windows": 0,
            "fraction_infinite": float(np.mean(
                ~np.isfinite(lower) | ~np.isfinite(upper)
            )),
        }


# ============================================================
# DRIVER
# ============================================================
def run_one_method(dataset_name, method_name, full_dataset, scaler, **kwargs):
    print(f"\n{'=' * 70}\n{dataset_name} :: {method_name}\n{'=' * 70}")
    run_fn = METHOD_FUNCS[method_name]
    start_time = time.time()
    alphaTraj, errSeq, y_u_preds, y_l_preds, y_true_vals = run_fn(full_dataset, **kwargs)
    time_taken = time.time() - start_time

    # Direct inverse affine mapping also preserves +/- infinity, which is
    # an intentional DtACI prediction-set boundary convention.
    offset = float(scaler.mean_[0])
    scale = float(scaler.scale_[0])
    y_u_inv = y_u_preds * scale + offset
    y_l_inv = y_l_preds * scale + offset
    y_true_inv = y_true_vals * scale + offset

    finite_cross = ((y_l_inv > y_u_inv) &
                    np.isfinite(y_l_inv) & np.isfinite(y_u_inv))
    any_unbounded = not (np.isfinite(y_u_inv).all() and np.isfinite(y_l_inv).all())
    if not finite_cross.any() and not any_unbounded:
        # Preserve the original metrics helper for ordinary finite intervals.
        metrics = summarize_all_metrics(
            y_true_inv, y_l_inv, y_u_inv,
            alpha=ALPHA, w=LOCAL_WINDOW, eta=CWC_ETA,
            time_taken=time_taken,
        )
        metrics["n_finite_crossed"] = 0
        metrics["n_excluded_windows"] = 0
        metrics["fraction_infinite"] = 0.0
    else:
        # No point or local window is discarded, even when bounds cross.
        metrics = metrics_all_points_fallback(
            y_true_inv, y_l_inv, y_u_inv, time_taken
        )
        print(f"[INFO] {method_name}: all {len(y_true_inv)} prediction points "
              f"and {metrics['n_local_windows']} local windows included; "
              f"crossings={metrics['n_finite_crossed']}, "
              f"infinite-bound fraction={metrics['fraction_infinite']:.4f}. "
              "LCWC unavailable for exceptional intervals.")

    print(f"[{dataset_name}/{method_name}] " +
          " | ".join(f"{k}={v:.4f}" for k, v in metrics.items()
                      if isinstance(v, float) and k != "time_sec"))
    print(f"[{dataset_name}/{method_name}] time_sec={time_taken:.2f}")

    if SAVE_TRAJECTORIES:
        os.makedirs(TRAJECTORY_DIR, exist_ok=True)
        max_len = max(len(alphaTraj), len(errSeq), len(y_u_inv), len(y_l_inv), len(y_true_inv))
        traj_df = pd.DataFrame({
            "step": np.arange(max_len),
            "alphaTrajectory": pad(alphaTraj, max_len),
            "errSeq": pad(errSeq, max_len),
            "y_u_pred": pad(y_u_inv, max_len),
            "y_l_pred": pad(y_l_inv, max_len),
            "y_true": pad(y_true_inv, max_len),
        })
        safe_ds = os.path.splitext(os.path.basename(dataset_name))[0]
        traj_df.to_csv(os.path.join(TRAJECTORY_DIR, f"{safe_ds}__{method_name}.csv"), index=False)

    return metrics


def main(dataset_paths):
    if not dataset_paths:
        print("No datasets provided. Edit DATASET_PATHS at the top of this "
              "script, or pass paths with --data path1.csv path2.csv ...")
        sys.exit(1)

    for path in dataset_paths:
        if not os.path.isfile(path):
            raise FileNotFoundError(f"Missing location dataset: {path}")
    all_rows = []
    for dataset_path in dataset_paths:
        print(f"\n\n########## Loading dataset: {dataset_path} ##########")
        full_dataset, scaler, y_raw, y_scaled = load_dataset(dataset_path, seq_len=SEQ_LEN)
        print(f"Series length: {len(y_raw)}, usable (t - seq_len) pairs: {len(full_dataset)}")

        if len(full_dataset) <= TINIT:
            print(f"WARNING: dataset {dataset_path} has fewer than TINIT={TINIT} "
                  f"usable points ({len(full_dataset)}). Skipping.")
            continue

        for method_name in METHODS_TO_RUN:
            kwargs = dict(alpha=ALPHA, tinit=TINIT, splitSize=SPLIT)
            if method_name in ("ACI", "LACI"):
                kwargs["gamma"] = GAMMA
            elif method_name == "DtACI":
                kwargs["gamma_candidates"] = DTACI_GAMMA_CANDIDATES
                kwargs["interval_size"] = DTACI_INTERVAL_SIZE
            elif method_name == "AgACI":
                kwargs["gamma_candidates"] = AGACI_GAMMA_CANDIDATES
                kwargs["eps"] = AGACI_EPS

            metrics = run_one_method(dataset_path, method_name, full_dataset, scaler, **kwargs)
            row = {"dataset": os.path.basename(dataset_path), "method": method_name}
            row.update(metrics)
            all_rows.append(row)

            pd.DataFrame(all_rows).to_csv(OUTPUT_CSV, index=False)

    results_df = pd.DataFrame(all_rows)
    col_order = ["dataset", "method", "PICP", "MPIW", "MLPICP", "LPICE", "LPIA",
                 "LMPIW", "LCWC", "STD_LE", "n_local_windows", "time_sec"]
    col_order = [c for c in col_order if c in results_df.columns]
    results_df = results_df[col_order + [c for c in results_df.columns if c not in col_order]]
    results_df.to_csv(OUTPUT_CSV, index=False)

    print(f"\nAll results written to: {OUTPUT_CSV}")
    if SAVE_TRAJECTORIES:
        print(f"Per-step trajectories written to: {TRAJECTORY_DIR}/")
    print("\n", results_df.to_string(index=False))
    return results_df


if __name__ == "__main__":
    dataset_paths = DATASET_PATHS
    main(dataset_paths)


device: cuda


########## Loading dataset: ../Datasets/Location1.csv ##########
Series length: 10000, usable (t - seq_len) pairs: 9904

../Datasets/Location1.csv :: AgACI
[AgACI] t=5000 err=0 interval=[-1.129, -1.026]
[AgACI] t=5200 err=0 interval=[-1.308, -1.202]
[AgACI] t=5400 err=0 interval=[0.435, 0.557]
[AgACI] t=5600 err=0 interval=[-1.229, -1.155]
[AgACI] t=5800 err=0 interval=[-0.200, -0.165]
[AgACI] t=6000 err=0 interval=[-0.771, -0.723]
[AgACI] t=6200 err=0 interval=[-0.116, -0.051]
[AgACI] t=6400 err=0 interval=[1.190, 1.202]
[AgACI] t=6600 err=0 interval=[1.700, 1.786]
[AgACI] t=6800 err=0 interval=[-0.127, -0.104]
[AgACI] t=7000 err=0 interval=[1.927, 2.053]
[AgACI] t=7200 err=0 interval=[-0.522, -0.445]
[AgACI] t=7400 err=0 interval=[0.832, 0.872]
[AgACI] t=7600 err=0 interval=[1.688, 1.780]
[AgACI] t=7800 err=0 interval=[0.410, 0.480]
[AgACI] t=8000 err=1 interval=[1.862, 2.028]
[AgACI] t=8200 err=0 interval=[-1.433, -1.375]
[AgACI] t=8400 err=0 interval=[-0.400, -0.329]

In [ ]:
# For plant datasets 

"""Plant 1/2 solar generation: ACI, LACI, DtACI and AgACI.

Requires existing Helper_Scripts and solar generation CSVs containing DAILY_YIELD.
Uses all source rows, including valid zeros; missing/nonfinite data raise errors.
""" 


# Research audit note:
# DtACI implements deterministic Algorithm 2 of Gibbs & Candes (2023 preprint).
# AgACI implements Algorithm 1 endpoints and a Python BOA-style gradient trick.
# Exact parity with R OPERA's BOA numerical details is NOT established.
# Original time-series CSV flattening and model helpers still require verification.
# Infinite conformal intervals are preserved. The robust metrics branch
# handles them without pretending they were finite forecasts.

# For Plant 1 and Plant 2
import os
import sys
import time
import glob
import random
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

from Helper_Scripts.calculate_metrics import (
    summarize_all_metrics,
    WINDOW as LOCAL_WINDOW,
    CWC_ETA,
)

from Helper_Scripts.models_TLMR import LSTMQuantileReg

# ============================================================
# GLOBAL CONFIG
# ============================================================
DATASET_PATHS = [
    "../Datasets/Plant_1_Generation_Data.csv",
    "../Datasets/Plant_2_Generation_Data.csv",
]

OUTPUT_CSV = "./results_plants_1_2.csv"
TRAJECTORY_DIR = "./trajectories/GRU"
SAVE_TRAJECTORIES = True

SEQ_LEN    = 96
EPOCHS     = 200
LR         = 0.001
SEED       = 42
BATCH_SIZE = 32

ALPHA    = 0.1
TINIT    = 4000
SPLIT    = 0.5
GAMMA    = 0.001

DTACI_GAMMA_CANDIDATES = [0.001, 0.002, 0.004, 0.008, 0.016, 0.032, 0.064, 0.128]
DTACI_INTERVAL_SIZE    = 500

AGACI_GAMMA_CANDIDATES = [0.001, 0.002, 0.004, 0.008, 0.0160]
AGACI_EPS = 0.001

ALPHA_MIN = 1e-6
ALPHA_MAX = 1.0 - 1e-6

METHODS_TO_RUN = ["ACI", "LACI", "DtACI", "AgACI"]


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)


# ============================================================
# MODEL: LSTM quantile regression
# ============================================================

def quantile_loss(y_true, y_pred, q):
    diff = y_true - y_pred
    return torch.mean(torch.maximum(q * diff, (q - 1) * diff))


# ============================================================
# DATA LOADING
# ============================================================
class SeqDataset(Dataset):
    def __init__(self, series, seq_len):
        self.X, self.y = [], []
        for i in range(len(series) - seq_len):
            self.X.append(series[i:i + seq_len])
            self.y.append(series[i + seq_len])
        self.X, self.y = np.array(self.X, dtype=np.float32), np.array(self.y, dtype=np.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.float32)


# For Solar generation datasets:
def load_dataset(filepath, seq_len=SEQ_LEN):
    """Use all DAILY_YIELD rows, including zero values; no future-data scaling leakage.

    Missing and infinite values are rejected instead of silently dropped, so
    row positions and temporal spacing are never changed during preprocessing.
    """
    df = pd.read_csv(filepath)
    if "DAILY_YIELD" not in df.columns:
        raise ValueError(f"{filepath}: missing DAILY_YIELD column; found {list(df.columns)}")
    y = pd.to_numeric(df["DAILY_YIELD"], errors="raise").to_numpy(dtype=np.float64)
    y = y[:40000]
    y = y[::4]
    if not np.isfinite(y).all():
        raise ValueError(f"{filepath}: DAILY_YIELD has NaN/Inf; fix the source explicitly")
    fit_end = TINIT + seq_len
    if len(y) <= fit_end:
        raise ValueError(f"{filepath}: need >{fit_end} observations, got {len(y)}")
    scaler = StandardScaler()
    scaler.fit(y[:fit_end].reshape(-1, 1))
    y_scaled = scaler.transform(y.reshape(-1, 1)).ravel()
    print(f"{Path(filepath).name}: retained all {len(y)} source rows (zeros included)")
    return SeqDataset(y_scaled, seq_len), scaler, y, y_scaled


# ============================================================
# SHARED HELPERS
# ============================================================
def pad(arr, length):
    arr = np.array(arr, dtype=np.float64)
    if len(arr) < length:
        arr = np.concatenate([arr, np.full(length - len(arr), np.nan)])
    return arr


def scoring_function_v1(error_history, magnitude_history):
    if len(error_history) == 0:
        return 0.0
    n_hist = len(error_history)
    exponents = np.arange(n_hist - 1, -1, -1, dtype=np.float64)
    recency_w = 0.85 ** exponents
    #To prevent divding by zero
    mag_w = np.array(magnitude_history, dtype=np.float64) + 1e-4
    joint_w = recency_w * mag_w
    joint_w = joint_w / joint_w.sum()
    return float(np.dot(joint_w, error_history))


def pinball_loss(beta, theta, alpha=ALPHA):
    diff = beta - theta
    return alpha * diff - min(0.0, diff)


def beta_from_scores(score_t, scores_cal):
    beta_t = 1.0 - (np.sum(scores_cal <= score_t) / (len(scores_cal) + 1))
    return float(np.clip(beta_t, 0.0, 1.0))


def _maybe_train(t, tinit, X_train, y_train, model_lower, model_upper, opt_lower, opt_upper, quantiles):
    if t == tinit:
        current_epochs = EPOCHS
    elif (t - tinit) % 6 == 0:
        current_epochs = 1
    else:
        current_epochs = 0

    if current_epochs > 0:
        X_train_tensor = torch.tensor(X_train.reshape(-1, SEQ_LEN), dtype=torch.float32, device=device)
        y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32, device=device)
        train_ds = TensorDataset(X_train_tensor, y_train_tensor)
        train_loader = DataLoader(train_ds, batch_size=min(BATCH_SIZE, len(train_ds)), shuffle=True)

        model_lower.train()
        for _ in range(current_epochs):
            for X_b, y_b in train_loader:
                loss = quantile_loss(y_b, model_lower(X_b), quantiles[0])
                opt_lower.zero_grad(); loss.backward(); opt_lower.step()

        model_upper.train()
        for _ in range(current_epochs):
            for X_b, y_b in train_loader:
                loss = quantile_loss(y_b, model_upper(X_b), quantiles[1])
                opt_upper.zero_grad(); loss.backward(); opt_upper.step()

    return current_epochs


def _fresh_models():
    model_lower = GRUQuantileReg().to(device)
    model_upper = GRUQuantileReg().to(device)
    opt_lower = torch.optim.Adam(model_lower.parameters(), lr=LR)
    opt_upper = torch.optim.Adam(model_upper.parameters(), lr=LR)
    return model_lower, model_upper, opt_lower, opt_upper







def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def interval_error(y, lower, upper):
    return float(not (lower <= y <= upper))


def conformal_radius(scores, alpha_t):
    """Split conformal order statistic; +/- infinity denotes full/empty set.

    Score can be signed (CQR); hence zero radius is NOT imposed.
    At alpha <= 0, set is the entire real line.
    At alpha >= 1, set is empty (represented by q=-inf).
    """
    scores = np.asarray(scores, dtype=np.float64)
    n = len(scores)
    if n == 0:
        raise ValueError("No calibration scores")
    if alpha_t <= 0.0:
        return float("inf")
    if alpha_t >= 1.0:
        return float("-inf")
    rank = int(np.ceil((n + 1) * (1 - alpha_t)))
    if rank > n:
        return float("inf")
    if rank < 1:
        return float("-inf")
    return float(np.partition(scores, rank - 1)[rank - 1])


def conformal_bounds(q_lo, q_hi, scores, alpha_t):
    q = conformal_radius(scores, alpha_t)
    if q == float("inf"):
        return float("-inf"), float("inf")
    if q == float("-inf"):
        return float("inf"), float("-inf")
    return float(q_lo - q), float(q_hi + q)


def predict_quantiles(model_lower, model_upper, X, device):
    x = torch.as_tensor(X, dtype=torch.float32, device=device)
    with torch.no_grad():
        model_lower.eval()
        model_upper.eval()
        lo = model_lower(x).detach().cpu().numpy().reshape(-1)
        hi = model_upper(x).detach().cpu().numpy().reshape(-1)
    return lo, hi


def time_step_data(ds, t, history, split):
    a = t - history
    n_train = int(round(history * split))
    if n_train < 1 or n_train >= history:
        raise ValueError("Require nonempty train and calibration sets")
    return np.arange(a, a+n_train), np.arange(a+n_train, t)

# ============================================================
# METHOD 2: ACI (Eq. 2/4, Gibbs & Candes 2021)
# ============================================================
def run_aci(full_dataset, alpha=ALPHA, gamma=GAMMA, tinit=TINIT, splitSize=SPLIT):
    set_seed(SEED)
    T = len(full_dataset)
    n_pred = T - tinit
    alphaTrajectory = np.zeros(n_pred, dtype=np.float32)
    adaptErrSeq = np.zeros(n_pred, dtype=np.float32)
    y_u_preds = np.zeros(n_pred, dtype=np.float32)
    y_l_preds = np.zeros(n_pred, dtype=np.float32)
    y_true_vals = np.zeros(n_pred, dtype=np.float32)

    alphat = alpha
    WINDOW_SIZE = tinit
    quantiles = [alpha / 2.0, 1.0 - alpha / 2.0]
    model_lower, model_upper, opt_lower, opt_upper = _fresh_models()

    for t in range(tinit, T):
        start_idx = t - WINDOW_SIZE
        current_window_idx = list(range(start_idx, t))
        n_train = int(round(splitSize * WINDOW_SIZE))
        trainPoints = current_window_idx[:n_train]
        calPoints = current_window_idx[n_train:]

        X_train = full_dataset.X[trainPoints].astype(np.float32)
        y_train = full_dataset.y[trainPoints].astype(np.float32)
        X_cal = full_dataset.X[calPoints].astype(np.float32)
        y_cal = full_dataset.y[calPoints].astype(np.float32)
        y_t = float(full_dataset.y[t])

        _maybe_train(t, tinit, X_train, y_train, model_lower, model_upper, opt_lower, opt_upper, quantiles)

        X_cal_tensor = torch.tensor(X_cal.reshape(-1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            model_lower.eval(); model_upper.eval()
            y_l_cal = model_lower(X_cal_tensor).cpu().numpy().reshape(-1)
            y_u_cal = model_upper(X_cal_tensor).cpu().numpy().reshape(-1)
        scores_cal = np.maximum(y_l_cal - y_cal, y_cal - y_u_cal)

        X_t_tensor = torch.tensor(full_dataset.X[t].reshape(1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            q_lo_t = float(model_lower(X_t_tensor).cpu().numpy()[0, 0])
            q_hi_t = float(model_upper(X_t_tensor).cpu().numpy()[0, 0])

        # Same adaptive quantile rule as the four-dataset ACI/LACI script.
        alphat = float(np.clip(alphat, ALPHA_MIN, ALPHA_MAX))
        level_adapt = float(np.clip(
            (1.0 - alphat) * (1.0 + 1.0 / len(calPoints)), 0.0, 1.0
        ))
        confQuantAdapt = float(np.quantile(scores_cal, level_adapt))
        y_l_adapt = q_lo_t - confQuantAdapt
        y_u_adapt = q_hi_t + confQuantAdapt
        # No artificial clipping of issued intervals.

        err_t = float(not (y_l_adapt <= y_t <= y_u_adapt))
        alphat = float(np.clip(alphat + gamma * (alpha - err_t), ALPHA_MIN, ALPHA_MAX))

        idx = t - tinit
        alphaTrajectory[idx] = alphat
        adaptErrSeq[idx] = err_t
        y_u_preds[idx] = y_u_adapt
        y_l_preds[idx] = y_l_adapt
        y_true_vals[idx] = y_t

        if t % 200 == 0:
            print(f"[ACI] t={t}, alphat={alphat:.4f}, err_t={err_t:.0f}")

    return alphaTrajectory, adaptErrSeq, y_u_preds, y_l_preds, y_true_vals


# ============================================================
# METHOD 3: LACI (user's local variant)
# ============================================================
def run_laci(full_dataset, alpha=ALPHA, gamma=GAMMA, tinit=TINIT, splitSize=SPLIT):
    set_seed(SEED)
    T = len(full_dataset)
    n_pred = T - tinit
    alphaTrajectory = np.zeros(n_pred, dtype=np.float32)
    adaptErrSeq = np.zeros(n_pred, dtype=np.float32)
    y_u_preds = np.zeros(n_pred, dtype=np.float32)
    y_l_preds = np.zeros(n_pred, dtype=np.float32)
    y_true_vals = np.zeros(n_pred, dtype=np.float32)

    alphat = alpha
    WINDOW_SIZE = tinit
    quantiles = [alpha / 2.0, 1.0 - alpha / 2.0]
    error_history, magnitude_history = [], []
    # Coverage of the intervals issued online, rather than retrospective
    # predictions reconstructed using the latest trained model.
    covered_history = []
    local_w = 24
    model_lower, model_upper, opt_lower, opt_upper = _fresh_models()

    for t in range(tinit, T):
        start_idx = t - WINDOW_SIZE
        current_window_idx = list(range(start_idx, t))
        n_train = int(round(splitSize * WINDOW_SIZE))
        trainPoints = current_window_idx[:n_train]
        calPoints = current_window_idx[n_train:]

        X_train = full_dataset.X[trainPoints].astype(np.float32)
        y_train = full_dataset.y[trainPoints].astype(np.float32)
        X_cal = full_dataset.X[calPoints].astype(np.float32)
        y_cal = full_dataset.y[calPoints].astype(np.float32)
        y_t = float(full_dataset.y[t])

        _maybe_train(t, tinit, X_train, y_train, model_lower, model_upper, opt_lower, opt_upper, quantiles)

        X_cal_tensor = torch.tensor(X_cal.reshape(-1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            model_lower.eval(); model_upper.eval()
            y_l_cal = model_lower(X_cal_tensor).cpu().numpy().reshape(-1)
            y_u_cal = model_upper(X_cal_tensor).cpu().numpy().reshape(-1)
        scores_cal = np.maximum(y_l_cal - y_cal, y_cal - y_u_cal)


        X_t_tensor = torch.tensor(full_dataset.X[t].reshape(1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            q_lo_t = float(model_lower(X_t_tensor).cpu().numpy()[0, 0])
            q_hi_t = float(model_upper(X_t_tensor).cpu().numpy()[0, 0])

        # Same adaptive quantile rule as the four-dataset ACI/LACI script.
        alphat = float(np.clip(alphat, ALPHA_MIN, ALPHA_MAX))
        level_adapt = float(np.clip(
            (1.0 - alphat) * (1.0 + 1.0 / len(calPoints)), 0.0, 1.0
        ))
        confQuantAdapt = float(np.quantile(scores_cal, level_adapt))
        y_l_adapt = q_lo_t - confQuantAdapt
        y_u_adapt = q_hi_t + confQuantAdapt
        # No artificial clipping of issued intervals.

        # Realized local coverage: use only coverage outcomes from intervals
        # actually issued at each past prediction step (including this one).
        binary_err = float(not (y_l_adapt <= y_t <= y_u_adapt))
        covered_history.append(1.0 - binary_err)

        if len(covered_history) >= local_w:
            lpicp = float(np.mean(covered_history[-local_w:]))
            magnitude = float(max((1.0 - alpha) - lpicp, 0.0))
        else:
            # The first w-1 steps have no full online local window.
            # Define their local error as zero, while recording their
            # actual coverage errors for the weighted update.
            magnitude = 0.0

        error_history.append(binary_err)
        magnitude_history.append(magnitude)
        # scoring_function_v1 uses decay 0.95 and epsilon 1e-5.
        weighted_err = scoring_function_v1(error_history, magnitude_history)

        alphat = float(np.clip(alphat + gamma * (alpha - weighted_err), ALPHA_MIN, ALPHA_MAX))

        idx = t - tinit
        alphaTrajectory[idx] = alphat
        adaptErrSeq[idx] = binary_err
        y_u_preds[idx] = y_u_adapt
        y_l_preds[idx] = y_l_adapt
        y_true_vals[idx] = y_t

        if t % 200 == 0:
            print(f"[LACI] t={t}, alphat={alphat:.4f}, weighted_err={weighted_err:.4f}")

    return alphaTrajectory, adaptErrSeq, y_u_preds, y_l_preds, y_true_vals


# ============================================================
# METHOD 4: DtACI
# ============================================================
def run_dtaci(full_dataset, alpha=ALPHA, tinit=TINIT, splitSize=SPLIT,
              gamma_candidates=None, interval_size=DTACI_INTERVAL_SIZE):
    """Gibbs & Candes 2023, Algorithm 2 (deterministic DtACI).

    Expert states are PROJECTED to [ALPHA_MIN, ALPHA_MAX]. Quantile conventions:
    alpha <= 0 -> R, alpha >= 1 -> empty set.
    Separate LSTMs are trained for this method; no other method's model is used.
    """
    if gamma_candidates is None:
        gamma_candidates = DTACI_GAMMA_CANDIDATES
    gammas = np.asarray(gamma_candidates, dtype=np.float64)
    k = len(gammas)
    if k < 1 or interval_size <= 0:
        raise ValueError("Invalid DtACI gamma candidates or interval size")
    set_seed(SEED)
    T = len(full_dataset)
    n_pred = T-tinit
    alpha_traj = np.empty(n_pred, np.float64)
    errors = np.empty(n_pred, np.float64)
    upper = np.empty(n_pred, np.float64)
    lower = np.empty(n_pred, np.float64)
    truth = np.empty(n_pred, np.float64)
    expert_alpha = np.full(k, alpha, np.float64)
    # Normalized weights, equivalent to the paper's unnormalized recursion.
    weights = np.full(k, 1/k, np.float64)
    sigma = 1.0 / (2.0 * interval_size)
    # Section 3.1 fixed heuristic: E[loss^2] approx (1-alpha)^2 alpha^2 / 3.
    eta = np.sqrt(3 * (np.log(k*interval_size)+2) /
                  ((1-alpha)**2 * alpha**2)) / interval_size
    quantiles = [alpha/2, 1-alpha/2]
    mlo, mhi, olo, ohi = _fresh_models()
    for t in range(tinit, T):
        tr, cal = time_step_data(full_dataset, t, tinit, splitSize)
        _maybe_train(t, tinit, full_dataset.X[tr], full_dataset.y[tr],
                     mlo, mhi, olo, ohi, quantiles)
        cal_l, cal_u = predict_quantiles(mlo, mhi, full_dataset.X[cal], device)
        scores = np.maximum(cal_l-full_dataset.y[cal], full_dataset.y[cal]-cal_u)
        ql, qu = predict_quantiles(mlo, mhi, full_dataset.X[t:t+1], device)
        ql, qu = float(ql[0]), float(qu[0])
        yt = float(full_dataset.y[t])
        beta = beta_from_scores(max(ql-yt, yt-qu), scores)

        probs = weights / weights.sum()
        alpha_out = float(np.clip(np.dot(probs, expert_alpha), ALPHA_MIN, ALPHA_MAX))
        lo_t, hi_t = conformal_bounds(ql, qu, scores, alpha_out)
        err_t = interval_error(yt, lo_t, hi_t)

        # Expert losses for past expert states; update AFTER forecasting.
        expert_errors = np.array([
            interval_error(yt, *conformal_bounds(ql, qu, scores, a))
            for a in expert_alpha
        ], dtype=np.float64)
        losses = np.array([pinball_loss(beta, a, alpha) for a in expert_alpha])
        logw = np.log(weights) - eta*losses
        logw -= np.max(logw)
        wbar = np.exp(logw)
        wbar /= wbar.sum()
        weights = (1 - sigma)*wbar + sigma/k
        expert_alpha = np.clip(
            expert_alpha + gammas * (alpha - expert_errors),
            ALPHA_MIN, ALPHA_MAX
        )

        j = t-tinit
        alpha_traj[j], errors[j] = alpha_out, err_t
        lower[j], upper[j], truth[j] = lo_t, hi_t, yt
        if t % 200 == 0:
            print(f"[DtACI] t={t} alpha={alpha_out:.4f} err={err_t:.0f}")
    return alpha_traj, errors, upper, lower, truth


# ============================================================
# METHOD 5: AgACI
# ============================================================
class BOAEndpoint:
    """Adaptive Bernstein expert aggregation with pinball gradient trick.

    This is a transparent Python BOA-style implementation, not a verified
    numerical reproduction of the R OPERA package used by Zaffran et al.
    Separate instances are maintained for each endpoint.
    """
    def __init__(self, n_experts, tau, eps):
        self.k = n_experts
        self.tau = tau
        self.eps = eps
        self.p = np.full(n_experts, 1.0/n_experts)
        self.eta = np.zeros(n_experts)
        self.L = np.zeros(n_experts)
        self.squared = np.zeros(n_experts)
        self.max_abs = np.zeros(n_experts)

    def forecast(self, expert_values):
        return float(self.p @ expert_values)

    def update(self, expert_values, aggregated, observed):
        # d/d forecast of pinball loss rho_tau(y-forecast).
        gradient = float(observed < aggregated) - self.tau
        regrets = gradient * (expert_values - aggregated)
        self.squared += regrets**2
        self.max_abs = np.maximum(self.max_abs, np.abs(regrets))
        E = 2.0**(np.ceil(np.log2(self.max_abs + self.eps)) + 1.0)
        self.L += 0.5*(regrets*(1 + self.eta*regrets) +
                       E*(self.eta*regrets > 0.5))
        candidate = np.full(self.k, np.inf)
        active = self.squared > 0
        candidate[active] = np.sqrt(np.log(self.k)/self.squared[active])
        self.eta = np.minimum(1.0/E, candidate)
        logw = np.log(self.eta) - self.eta*self.L
        logw -= np.max(logw)
        w = np.exp(logw)
        self.p = w/w.sum()


def run_agaci(full_dataset, alpha=ALPHA, tinit=TINIT, splitSize=SPLIT,
              gamma_candidates=None, eps=AGACI_EPS):
    """Zaffran et al. ICML 2022 AgACI: separate endpoint BOA aggregation.

    At each t, every gamma expert creates an ACI interval; the lower and
    upper endpoint forecasts are aggregated with *separate* BOA weights.
    Infinite expert bounds are replaced by predefined finite training-only
    thresholds, as permitted by Algorithm 1's M^(l), M^(u).
    """
    if gamma_candidates is None:
        gamma_candidates = AGACI_GAMMA_CANDIDATES
    gammas = np.asarray(gamma_candidates, dtype=np.float64)
    k = len(gammas)
    if k < 2:
        raise ValueError("BOA aggregation needs at least two experts")
    set_seed(SEED)
    T = len(full_dataset)
    n_pred = T-tinit
    alpha_traj = np.full(n_pred, np.nan, np.float64)  # no single AgACI alpha!
    errors = np.empty(n_pred, np.float64)
    upper = np.empty(n_pred, np.float64)
    lower = np.empty(n_pred, np.float64)
    truth = np.empty(n_pred, np.float64)
    expert_alpha = np.full(k, alpha, np.float64)
    lower_boa = BOAEndpoint(k, alpha/2, eps)
    upper_boa = BOAEndpoint(k, 1-alpha/2, eps)
    quantiles = [alpha/2, 1-alpha/2]
    mlo, mhi, olo, ohi = _fresh_models()

    # Predeclare finite endpoints from TRAINING-ONLY raw-scale standardized
    # history. These are a stated experimental choice for M^(l), M^(u).
    initial = np.asarray(full_dataset.y[:tinit], dtype=float)
    yrange = float(np.ptp(initial))
    margin = 5.0 * max(yrange, 1e-8)
    m_lower = float(np.min(initial) - margin)
    m_upper = float(np.max(initial) + margin)

    for t in range(tinit, T):
        tr, cal = time_step_data(full_dataset, t, tinit, splitSize)
        _maybe_train(t, tinit, full_dataset.X[tr], full_dataset.y[tr],
                     mlo, mhi, olo, ohi, quantiles)
        cal_l, cal_u = predict_quantiles(mlo, mhi, full_dataset.X[cal], device)
        scores = np.maximum(cal_l-full_dataset.y[cal], full_dataset.y[cal]-cal_u)
        ql, qu = predict_quantiles(mlo, mhi, full_dataset.X[t:t+1], device)
        ql, qu = float(ql[0]), float(qu[0])
        yt = float(full_dataset.y[t])

        raw_intervals = [conformal_bounds(ql, qu, scores, a)
                         for a in expert_alpha]
        expert_errors = np.array([
            interval_error(yt, lo, hi) for lo, hi in raw_intervals
        ])
        # BOA requires bounded expert predictions. These thresholds apply
        # only to AgACI's endpoint experts, not to DtACI/LACI predictions.
        expert_lower = np.array([
            np.clip(lo, m_lower, m_upper) for lo, hi in raw_intervals
        ], dtype=float)
        expert_upper = np.array([
            np.clip(hi, m_lower, m_upper) for lo, hi in raw_intervals
        ], dtype=float)
        pred_lower = lower_boa.forecast(expert_lower)
        pred_upper = upper_boa.forecast(expert_upper)
        actual_err = interval_error(yt, pred_lower, pred_upper)

        # Observe target, update each BOA endpoint separately.
        lower_boa.update(expert_lower, pred_lower, yt)
        upper_boa.update(expert_upper, pred_upper, yt)
        # Each expert follows its own ACI feedback, not the aggregate error.
        expert_alpha = np.clip(
            expert_alpha + gammas * (alpha - expert_errors),
            ALPHA_MIN, ALPHA_MAX
        )

        j = t-tinit
        errors[j] = actual_err
        lower[j], upper[j], truth[j] = pred_lower, pred_upper, yt
        if t % 200 == 0:
            print(f"[AgACI] t={t} err={actual_err:.0f} "
                  f"interval=[{pred_lower:.3f}, {pred_upper:.3f}]")
    return alpha_traj, errors, upper, lower, truth


METHOD_FUNCS = {
   
    "ACI": run_aci,
    "LACI": run_laci,
    "DtACI": run_dtaci,
    "AgACI": run_agaci,
}


# ============================================================
# ALL-POINT EVALUATION (NO EXCLUDED INTERVALS OR LOCAL WINDOWS)
# ============================================================
def metrics_all_points(y_true, lower, upper, elapsed, alpha=ALPHA,
                       window=LOCAL_WINDOW):
    """Compute coverage and width metrics over every online forecast.

    A crossed interval is empty and thus contributes coverage 0 and width 0.
    An interval (-inf, inf) covers every finite target and has infinite width.
    Every overlapping full local window is retained.
    """
    y_true = np.asarray(y_true, dtype=np.float64)
    lower = np.asarray(lower, dtype=np.float64)
    upper = np.asarray(upper, dtype=np.float64)
    covered = (lower <= y_true) & (y_true <= upper)
    crossed = lower > upper
    width = np.maximum(upper - lower, 0.0)
    # np.maximum(-inf, 0) correctly gives zero width for empty prediction sets.
    n_windows = max(0, len(y_true) - window + 1)
    window_cov = np.array([
        float(np.mean(covered[i:i + window])) for i in range(n_windows)
    ], dtype=np.float64)
    window_width = np.array([
        float(np.mean(width[i:i + window])) for i in range(n_windows)
    ], dtype=np.float64)
    local_error = np.maximum((1.0 - alpha) - window_cov, 0.0)
    finite_cross = crossed & np.isfinite(lower) & np.isfinite(upper)
    return {
        "PICP": float(np.mean(covered)),
        "MPIW": float(np.mean(width)),
        "MLPICP": float(np.mean(window_cov)) if n_windows else np.nan,
        "LPICE": float(np.mean(local_error)) if n_windows else np.nan,
        "LPIA": float(np.mean(window_cov >= 1.0 - alpha)) if n_windows else np.nan,
        "LMPIW": float(np.mean(window_width)) if n_windows else np.nan,
        "LCWC": np.nan,  # External LCWC helper not specified for infinite/empty sets.
        "STD_LE": float(np.std(local_error)) if n_windows else np.nan,
        "n_local_windows": n_windows,
        "time_sec": float(elapsed),
        "n_finite_crossed": int(np.sum(finite_cross)),
        "n_excluded_windows": 0,
        "fraction_infinite": float(np.mean(~np.isfinite(lower) | ~np.isfinite(upper))),
    }


# ============================================================
# DRIVER
# ============================================================
def run_one_method(dataset_name, method_name, full_dataset, scaler, **kwargs):
    print(f"\n{'=' * 70}\n{dataset_name} :: {method_name}\n{'=' * 70}")
    run_fn = METHOD_FUNCS[method_name]
    start_time = time.time()
    alphaTraj, errSeq, y_u_preds, y_l_preds, y_true_vals = run_fn(full_dataset, **kwargs)
    time_taken = time.time() - start_time

    # Direct inverse affine mapping also preserves +/- infinity, which is
    # an intentional DtACI prediction-set boundary convention.
    offset = float(scaler.mean_[0])
    scale = float(scaler.scale_[0])
    y_u_inv = y_u_preds * scale + offset
    y_l_inv = y_l_preds * scale + offset
    y_true_inv = y_true_vals * scale + offset

    finite_cross = ((y_l_inv > y_u_inv) &
                    np.isfinite(y_l_inv) & np.isfinite(y_u_inv))
    all_finite = np.isfinite(y_u_inv).all() and np.isfinite(y_l_inv).all()
    if all_finite and not finite_cross.any():
        metrics = summarize_all_metrics(
            y_true_inv, y_l_inv, y_u_inv,
            alpha=ALPHA, w=LOCAL_WINDOW, eta=CWC_ETA,
            time_taken=time_taken,
        )
        metrics["n_finite_crossed"] = 0
        metrics["n_excluded_windows"] = 0
        metrics["fraction_infinite"] = 0.0
    else:
        metrics = metrics_all_points(y_true_inv, y_l_inv, y_u_inv, time_taken)
        print(f"[INFO] {method_name}: full evaluation retained every forecast and "
              f"every local window; crossed={metrics['n_finite_crossed']}, "
              f"unbounded_fraction={metrics['fraction_infinite']:.4f}. "
              "LCWC is NaN in this exceptional-interval branch.")

    print(f"[{dataset_name}/{method_name}] " +
          " | ".join(f"{k}={v:.4f}" for k, v in metrics.items()
                      if isinstance(v, float) and k != "time_sec"))
    print(f"[{dataset_name}/{method_name}] time_sec={time_taken:.2f}")

    if SAVE_TRAJECTORIES:
        os.makedirs(TRAJECTORY_DIR, exist_ok=True)
        max_len = max(len(alphaTraj), len(errSeq), len(y_u_inv), len(y_l_inv), len(y_true_inv))
        traj_df = pd.DataFrame({
            "step": np.arange(max_len),
            "alphaTrajectory": pad(alphaTraj, max_len),
            "errSeq": pad(errSeq, max_len),
            "y_u_pred": pad(y_u_inv, max_len),
            "y_l_pred": pad(y_l_inv, max_len),
            "y_true": pad(y_true_inv, max_len),
        })
        safe_ds = os.path.splitext(os.path.basename(dataset_name))[0]
        traj_df.to_csv(os.path.join(TRAJECTORY_DIR, f"{safe_ds}__{method_name}.csv"), index=False)

    return metrics


def main(dataset_paths):
    if not dataset_paths:
        print("No datasets provided. Edit DATASET_PATHS at the top of this "
              "script, or pass paths with --data path1.csv path2.csv ...")
        sys.exit(1)

    for path in dataset_paths:
        if not os.path.isfile(path):
            raise FileNotFoundError(f"Missing solar generation dataset: {path}")
    all_rows = []
    for dataset_path in dataset_paths:
        print(f"\n\n########## Loading dataset: {dataset_path} ##########")
        full_dataset, scaler, y_raw, y_scaled = load_dataset(dataset_path, seq_len=SEQ_LEN)
        print(f"Series length: {len(y_raw)}, usable (t - seq_len) pairs: {len(full_dataset)}")

        if len(full_dataset) <= TINIT:
            print(f"WARNING: dataset {dataset_path} has fewer than TINIT={TINIT} "
                  f"usable points ({len(full_dataset)}). Skipping.")
            continue

        for method_name in METHODS_TO_RUN:
            kwargs = dict(alpha=ALPHA, tinit=TINIT, splitSize=SPLIT)
            if method_name in ("ACI", "LACI"):
                kwargs["gamma"] = GAMMA
            elif method_name == "DtACI":
                kwargs["gamma_candidates"] = DTACI_GAMMA_CANDIDATES
                kwargs["interval_size"] = DTACI_INTERVAL_SIZE
            elif method_name == "AgACI":
                kwargs["gamma_candidates"] = AGACI_GAMMA_CANDIDATES
                kwargs["eps"] = AGACI_EPS

            metrics = run_one_method(dataset_path, method_name, full_dataset, scaler, **kwargs)
            row = {"dataset": os.path.basename(dataset_path), "method": method_name}
            row.update(metrics)
            all_rows.append(row)

            pd.DataFrame(all_rows).to_csv(OUTPUT_CSV, index=False)

    results_df = pd.DataFrame(all_rows)
    col_order = ["dataset", "method", "PICP", "MPIW", "MLPICP", "LPICE", "LPIA",
                 "LMPIW", "LCWC", "STD_LE", "n_local_windows", "time_sec"]
    col_order = [c for c in col_order if c in results_df.columns]
    results_df = results_df[col_order + [c for c in results_df.columns if c not in col_order]]
    results_df.to_csv(OUTPUT_CSV, index=False)

    print(f"\nAll results written to: {OUTPUT_CSV}")
    if SAVE_TRAJECTORIES:
        print(f"Per-step trajectories written to: {TRAJECTORY_DIR}/")
    print("\n", results_df.to_string(index=False))
    return results_df


if __name__ == "__main__":
    dataset_paths = DATASET_PATHS
    main(dataset_paths)

device: cuda


########## Loading dataset: ../Datasets/Plant_1_Generation_Data.csv ##########
Plant_1_Generation_Data.csv: retained all 10000 source rows (zeros included)
Series length: 10000, usable (t - seq_len) pairs: 9904

../Datasets/Plant_1_Generation_Data.csv :: ACI
[ACI] t=4000, alphat=0.1001, err_t=0
[ACI] t=4200, alphat=0.0841, err_t=0
[ACI] t=4400, alphat=0.1041, err_t=0
[ACI] t=4600, alphat=0.1101, err_t=0
[ACI] t=4800, alphat=0.1061, err_t=0
[ACI] t=5000, alphat=0.1181, err_t=0
[ACI] t=5200, alphat=0.1041, err_t=0
[ACI] t=5400, alphat=0.1101, err_t=0
[ACI] t=5600, alphat=0.1081, err_t=1
[ACI] t=5800, alphat=0.1101, err_t=0
[ACI] t=6000, alphat=0.1241, err_t=0
[ACI] t=6200, alphat=0.0891, err_t=0
[ACI] t=6400, alphat=0.0891, err_t=0
[ACI] t=6600, alphat=0.0971, err_t=1
[ACI] t=6800, alphat=0.0721, err_t=0


In [ ]:
# For hydropower 

"""Hydropower daily peak generation: ACI, LACI, DtACI and AgACI.

Requires Helper_Scripts and a hydropower CSV containing both actual peak columns.
All daily records, including zero totals, are retained; missing/nonfinite inputs raise errors.
Six-step retraining and 24-step metrics mean DAYS for daily records.
""" 


# Research audit note:
# DtACI implements deterministic Algorithm 2 of Gibbs & Candes (2023 preprint).
# AgACI implements Algorithm 1 endpoints and a Python BOA-style gradient trick.
# Exact parity with R OPERA's BOA numerical details is NOT established.
# Hydropower source chronology and model helpers still require verification.
# Infinite conformal intervals are preserved. The robust metrics branch
# handles them without pretending they were finite forecasts.

# For hydropower dataset
import os
import sys
import time
import glob
import random
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

from Helper_Scripts.calculate_metrics import (
    summarize_all_metrics,
    WINDOW as LOCAL_WINDOW,
    CWC_ETA,
)

from Helper_Scripts.models_TLMR import LSTMQuantileReg

# ============================================================
# GLOBAL CONFIG
# ============================================================
DATASET_PATHS = [
    "../Datasets/hydropower_dataset.csv",
]

OUTPUT_CSV = "./results_hydropower.csv"
TRAJECTORY_DIR = "./trajectories/GRU"
SAVE_TRAJECTORIES = True

SEQ_LEN    = 96
EPOCHS     = 200
LR         = 0.001
SEED       = 42
BATCH_SIZE = 32

ALPHA    = 0.1
TINIT    = 200
SPLIT    = 0.5
GAMMA    = 0.001

DTACI_GAMMA_CANDIDATES = [0.001, 0.002, 0.004, 0.008, 0.016, 0.032, 0.064, 0.128]
DTACI_INTERVAL_SIZE    = 500

AGACI_GAMMA_CANDIDATES = [0.001, 0.002, 0.004, 0.008, 0.0160]
AGACI_EPS = 0.001

ALPHA_MIN = 1e-6
ALPHA_MAX = 1.0 - 1e-6

METHODS_TO_RUN = ["ACI", "LACI", "DtACI", "AgACI"]


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)


# ============================================================
# MODEL: LSTM quantile regression
# ============================================================

def quantile_loss(y_true, y_pred, q):
    diff = y_true - y_pred
    return torch.mean(torch.maximum(q * diff, (q - 1) * diff))


# ============================================================
# DATA LOADING
# ============================================================
class SeqDataset(Dataset):
    def __init__(self, series, seq_len):
        self.X, self.y = [], []
        for i in range(len(series) - seq_len):
            self.X.append(series[i:i + seq_len])
            self.y.append(series[i + seq_len])
        self.X, self.y = np.array(self.X, dtype=np.float32), np.array(self.y, dtype=np.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.float32)


# For Hydropower dataset:
def load_dataset(filepath, seq_len=SEQ_LEN):
    """Sum observed day/evening peaks, retaining all chronological rows.

    Zeros are valid observations. Missing/nonfinite inputs raise an error rather
    than shortening the time series and changing its temporal spacing.
    """
    df = pd.read_csv(filepath)
    day_col = "Actual Day Peak (MW)"
    eve_col = "Actual Eve Peak (MW)"
    missing = [col for col in (day_col, eve_col) if col not in df.columns]
    if missing:
        raise ValueError(f"{filepath}: missing columns {missing}; available {list(df.columns)}")

    day = pd.to_numeric(df[day_col], errors="raise").to_numpy(dtype=np.float64)
    evening = pd.to_numeric(df[eve_col], errors="raise").to_numpy(dtype=np.float64)
    if not (np.isfinite(day).all() and np.isfinite(evening).all()):
        raise ValueError(f"{filepath}: missing or infinite peak values; correct source data")
    y = day + evening
    if not np.isfinite(y).all():
        raise ValueError(f"{filepath}: nonfinite summed peak values")

    fit_end = TINIT + seq_len
    if len(y) <= fit_end:
        raise ValueError(f"{filepath}: need >{fit_end} observations, got {len(y)}")
    scaler = StandardScaler()
    scaler.fit(y[:fit_end].reshape(-1, 1))
    y_scaled = scaler.transform(y.reshape(-1, 1)).ravel()
    print(f"{Path(filepath).name}: retained all {len(y)} daily records (including zeros)")
    return SeqDataset(y_scaled, seq_len), scaler, y, y_scaled


# ============================================================
# SHARED HELPERS
# ============================================================
def pad(arr, length):
    arr = np.array(arr, dtype=np.float64)
    if len(arr) < length:
        arr = np.concatenate([arr, np.full(length - len(arr), np.nan)])
    return arr


def scoring_function_v1(error_history, magnitude_history):
    if len(error_history) == 0:
        return 0.0
    n_hist = len(error_history)
    exponents = np.arange(n_hist - 1, -1, -1, dtype=np.float64)
    recency_w = 0.85 ** exponents
    #To prevent divding by zero
    mag_w = np.array(magnitude_history, dtype=np.float64) + 1e-4
    joint_w = recency_w * mag_w
    joint_w = joint_w / joint_w.sum()
    return float(np.dot(joint_w, error_history))


def pinball_loss(beta, theta, alpha=ALPHA):
    diff = beta - theta
    return alpha * diff - min(0.0, diff)


def beta_from_scores(score_t, scores_cal):
    beta_t = 1.0 - (np.sum(scores_cal <= score_t) / (len(scores_cal) + 1))
    return float(np.clip(beta_t, 0.0, 1.0))


def _maybe_train(t, tinit, X_train, y_train, model_lower, model_upper, opt_lower, opt_upper, quantiles):
    if t == tinit:
        current_epochs = EPOCHS
    elif (t - tinit) % 6 == 0:
        current_epochs = 1
    else:
        current_epochs = 0

    if current_epochs > 0:
        X_train_tensor = torch.tensor(X_train.reshape(-1, SEQ_LEN), dtype=torch.float32, device=device)
        y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32, device=device)
        train_ds = TensorDataset(X_train_tensor, y_train_tensor)
        train_loader = DataLoader(train_ds, batch_size=min(BATCH_SIZE, len(train_ds)), shuffle=True)

        model_lower.train()
        for _ in range(current_epochs):
            for X_b, y_b in train_loader:
                loss = quantile_loss(y_b, model_lower(X_b), quantiles[0])
                opt_lower.zero_grad(); loss.backward(); opt_lower.step()

        model_upper.train()
        for _ in range(current_epochs):
            for X_b, y_b in train_loader:
                loss = quantile_loss(y_b, model_upper(X_b), quantiles[1])
                opt_upper.zero_grad(); loss.backward(); opt_upper.step()

    return current_epochs


def _fresh_models():
    model_lower = GRUQuantileReg().to(device)
    model_upper = GRUQuantileReg().to(device)
    opt_lower = torch.optim.Adam(model_lower.parameters(), lr=LR)
    opt_upper = torch.optim.Adam(model_upper.parameters(), lr=LR)
    return model_lower, model_upper, opt_lower, opt_upper







def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def interval_error(y, lower, upper):
    return float(not (lower <= y <= upper))


def conformal_radius(scores, alpha_t):
    """Split conformal order statistic; +/- infinity denotes full/empty set.

    Score can be signed (CQR); hence zero radius is NOT imposed.
    At alpha <= 0, set is the entire real line.
    At alpha >= 1, set is empty (represented by q=-inf).
    """
    scores = np.asarray(scores, dtype=np.float64)
    n = len(scores)
    if n == 0:
        raise ValueError("No calibration scores")
    if alpha_t <= 0.0:
        return float("inf")
    if alpha_t >= 1.0:
        return float("-inf")
    rank = int(np.ceil((n + 1) * (1 - alpha_t)))
    if rank > n:
        return float("inf")
    if rank < 1:
        return float("-inf")
    return float(np.partition(scores, rank - 1)[rank - 1])


def conformal_bounds(q_lo, q_hi, scores, alpha_t):
    q = conformal_radius(scores, alpha_t)
    if q == float("inf"):
        return float("-inf"), float("inf")
    if q == float("-inf"):
        return float("inf"), float("-inf")
    return float(q_lo - q), float(q_hi + q)


def predict_quantiles(model_lower, model_upper, X, device):
    x = torch.as_tensor(X, dtype=torch.float32, device=device)
    with torch.no_grad():
        model_lower.eval()
        model_upper.eval()
        lo = model_lower(x).detach().cpu().numpy().reshape(-1)
        hi = model_upper(x).detach().cpu().numpy().reshape(-1)
    return lo, hi


def time_step_data(ds, t, history, split):
    a = t - history
    n_train = int(round(history * split))
    if n_train < 1 or n_train >= history:
        raise ValueError("Require nonempty train and calibration sets")
    return np.arange(a, a+n_train), np.arange(a+n_train, t)

# ============================================================
# METHOD 2: ACI (Eq. 2/4, Gibbs & Candes 2021)
# ============================================================
def run_aci(full_dataset, alpha=ALPHA, gamma=GAMMA, tinit=TINIT, splitSize=SPLIT):
    set_seed(SEED)
    T = len(full_dataset)
    n_pred = T - tinit
    alphaTrajectory = np.zeros(n_pred, dtype=np.float32)
    adaptErrSeq = np.zeros(n_pred, dtype=np.float32)
    y_u_preds = np.zeros(n_pred, dtype=np.float32)
    y_l_preds = np.zeros(n_pred, dtype=np.float32)
    y_true_vals = np.zeros(n_pred, dtype=np.float32)

    alphat = alpha
    WINDOW_SIZE = tinit
    quantiles = [alpha / 2.0, 1.0 - alpha / 2.0]
    model_lower, model_upper, opt_lower, opt_upper = _fresh_models()

    for t in range(tinit, T):
        start_idx = t - WINDOW_SIZE
        current_window_idx = list(range(start_idx, t))
        n_train = int(round(splitSize * WINDOW_SIZE))
        trainPoints = current_window_idx[:n_train]
        calPoints = current_window_idx[n_train:]

        X_train = full_dataset.X[trainPoints].astype(np.float32)
        y_train = full_dataset.y[trainPoints].astype(np.float32)
        X_cal = full_dataset.X[calPoints].astype(np.float32)
        y_cal = full_dataset.y[calPoints].astype(np.float32)
        y_t = float(full_dataset.y[t])

        _maybe_train(t, tinit, X_train, y_train, model_lower, model_upper, opt_lower, opt_upper, quantiles)

        X_cal_tensor = torch.tensor(X_cal.reshape(-1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            model_lower.eval(); model_upper.eval()
            y_l_cal = model_lower(X_cal_tensor).cpu().numpy().reshape(-1)
            y_u_cal = model_upper(X_cal_tensor).cpu().numpy().reshape(-1)
        scores_cal = np.maximum(y_l_cal - y_cal, y_cal - y_u_cal)

        X_t_tensor = torch.tensor(full_dataset.X[t].reshape(1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            q_lo_t = float(model_lower(X_t_tensor).cpu().numpy()[0, 0])
            q_hi_t = float(model_upper(X_t_tensor).cpu().numpy()[0, 0])

        # Same adaptive quantile rule as the four-dataset ACI/LACI script.
        alphat = float(np.clip(alphat, ALPHA_MIN, ALPHA_MAX))
        level_adapt = float(np.clip(
            (1.0 - alphat) * (1.0 + 1.0 / len(calPoints)), 0.0, 1.0
        ))
        confQuantAdapt = float(np.quantile(scores_cal, level_adapt))
        y_l_adapt = q_lo_t - confQuantAdapt
        y_u_adapt = q_hi_t + confQuantAdapt
        # No artificial clipping of issued intervals.

        err_t = float(not (y_l_adapt <= y_t <= y_u_adapt))
        alphat = float(np.clip(alphat + gamma * (alpha - err_t), ALPHA_MIN, ALPHA_MAX))

        idx = t - tinit
        alphaTrajectory[idx] = alphat
        adaptErrSeq[idx] = err_t
        y_u_preds[idx] = y_u_adapt
        y_l_preds[idx] = y_l_adapt
        y_true_vals[idx] = y_t

        if t % 200 == 0:
            print(f"[ACI] t={t}, alphat={alphat:.4f}, err_t={err_t:.0f}")

    return alphaTrajectory, adaptErrSeq, y_u_preds, y_l_preds, y_true_vals


# ============================================================
# METHOD 3: LACI (user's local variant)
# ============================================================
def run_laci(full_dataset, alpha=ALPHA, gamma=GAMMA, tinit=TINIT, splitSize=SPLIT):
    set_seed(SEED)
    T = len(full_dataset)
    n_pred = T - tinit
    alphaTrajectory = np.zeros(n_pred, dtype=np.float32)
    adaptErrSeq = np.zeros(n_pred, dtype=np.float32)
    y_u_preds = np.zeros(n_pred, dtype=np.float32)
    y_l_preds = np.zeros(n_pred, dtype=np.float32)
    y_true_vals = np.zeros(n_pred, dtype=np.float32)

    alphat = alpha
    WINDOW_SIZE = tinit
    quantiles = [alpha / 2.0, 1.0 - alpha / 2.0]
    error_history, magnitude_history = [], []
    # Coverage of the intervals issued online, rather than retrospective
    # predictions reconstructed using the latest trained model.
    covered_history = []
    local_w = 24
    model_lower, model_upper, opt_lower, opt_upper = _fresh_models()

    for t in range(tinit, T):
        start_idx = t - WINDOW_SIZE
        current_window_idx = list(range(start_idx, t))
        n_train = int(round(splitSize * WINDOW_SIZE))
        trainPoints = current_window_idx[:n_train]
        calPoints = current_window_idx[n_train:]

        X_train = full_dataset.X[trainPoints].astype(np.float32)
        y_train = full_dataset.y[trainPoints].astype(np.float32)
        X_cal = full_dataset.X[calPoints].astype(np.float32)
        y_cal = full_dataset.y[calPoints].astype(np.float32)
        y_t = float(full_dataset.y[t])

        _maybe_train(t, tinit, X_train, y_train, model_lower, model_upper, opt_lower, opt_upper, quantiles)

        X_cal_tensor = torch.tensor(X_cal.reshape(-1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            model_lower.eval(); model_upper.eval()
            y_l_cal = model_lower(X_cal_tensor).cpu().numpy().reshape(-1)
            y_u_cal = model_upper(X_cal_tensor).cpu().numpy().reshape(-1)
        scores_cal = np.maximum(y_l_cal - y_cal, y_cal - y_u_cal)


        X_t_tensor = torch.tensor(full_dataset.X[t].reshape(1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            q_lo_t = float(model_lower(X_t_tensor).cpu().numpy()[0, 0])
            q_hi_t = float(model_upper(X_t_tensor).cpu().numpy()[0, 0])

        # Same adaptive quantile rule as the four-dataset ACI/LACI script.
        alphat = float(np.clip(alphat, ALPHA_MIN, ALPHA_MAX))
        level_adapt = float(np.clip(
            (1.0 - alphat) * (1.0 + 1.0 / len(calPoints)), 0.0, 1.0
        ))
        confQuantAdapt = float(np.quantile(scores_cal, level_adapt))
        y_l_adapt = q_lo_t - confQuantAdapt
        y_u_adapt = q_hi_t + confQuantAdapt
        # No artificial clipping of issued intervals.

        # Realized local coverage: use only coverage outcomes from intervals
        # actually issued at each past prediction step (including this one).
        binary_err = float(not (y_l_adapt <= y_t <= y_u_adapt))
        covered_history.append(1.0 - binary_err)

        if len(covered_history) >= local_w:
            lpicp = float(np.mean(covered_history[-local_w:]))
            magnitude = float(max((1.0 - alpha) - lpicp, 0.0))
        else:
            # The first w-1 steps have no full online local window.
            # Define their local error as zero, while recording their
            # actual coverage errors for the weighted update.
            magnitude = 0.0

        error_history.append(binary_err)
        magnitude_history.append(magnitude)
        # scoring_function_v1 uses decay 0.95 and epsilon 1e-5.
        weighted_err = scoring_function_v1(error_history, magnitude_history)

        alphat = float(np.clip(alphat + gamma * (alpha - weighted_err), ALPHA_MIN, ALPHA_MAX))

        idx = t - tinit
        alphaTrajectory[idx] = alphat
        adaptErrSeq[idx] = binary_err
        y_u_preds[idx] = y_u_adapt
        y_l_preds[idx] = y_l_adapt
        y_true_vals[idx] = y_t

        if t % 200 == 0:
            print(f"[LACI] t={t}, alphat={alphat:.4f}, weighted_err={weighted_err:.4f}")

    return alphaTrajectory, adaptErrSeq, y_u_preds, y_l_preds, y_true_vals


# ============================================================
# METHOD 4: DtACI
# ============================================================
def run_dtaci(full_dataset, alpha=ALPHA, tinit=TINIT, splitSize=SPLIT,
              gamma_candidates=None, interval_size=DTACI_INTERVAL_SIZE):
    """Gibbs & Candes 2023, Algorithm 2 (deterministic DtACI).

    Expert states are PROJECTED to [ALPHA_MIN, ALPHA_MAX]. Quantile conventions:
    alpha <= 0 -> R, alpha >= 1 -> empty set.
    Separate LSTMs are trained for this method; no other method's model is used.
    """
    if gamma_candidates is None:
        gamma_candidates = DTACI_GAMMA_CANDIDATES
    gammas = np.asarray(gamma_candidates, dtype=np.float64)
    k = len(gammas)
    if k < 1 or interval_size <= 0:
        raise ValueError("Invalid DtACI gamma candidates or interval size")
    set_seed(SEED)
    T = len(full_dataset)
    n_pred = T-tinit
    alpha_traj = np.empty(n_pred, np.float64)
    errors = np.empty(n_pred, np.float64)
    upper = np.empty(n_pred, np.float64)
    lower = np.empty(n_pred, np.float64)
    truth = np.empty(n_pred, np.float64)
    expert_alpha = np.full(k, alpha, np.float64)
    # Normalized weights, equivalent to the paper's unnormalized recursion.
    weights = np.full(k, 1/k, np.float64)
    sigma = 1.0 / (2.0 * interval_size)
    # Section 3.1 fixed heuristic: E[loss^2] approx (1-alpha)^2 alpha^2 / 3.
    eta = np.sqrt(3 * (np.log(k*interval_size)+2) /
                  ((1-alpha)**2 * alpha**2)) / interval_size
    quantiles = [alpha/2, 1-alpha/2]
    mlo, mhi, olo, ohi = _fresh_models()
    for t in range(tinit, T):
        tr, cal = time_step_data(full_dataset, t, tinit, splitSize)
        _maybe_train(t, tinit, full_dataset.X[tr], full_dataset.y[tr],
                     mlo, mhi, olo, ohi, quantiles)
        cal_l, cal_u = predict_quantiles(mlo, mhi, full_dataset.X[cal], device)
        scores = np.maximum(cal_l-full_dataset.y[cal], full_dataset.y[cal]-cal_u)
        ql, qu = predict_quantiles(mlo, mhi, full_dataset.X[t:t+1], device)
        ql, qu = float(ql[0]), float(qu[0])
        yt = float(full_dataset.y[t])
        beta = beta_from_scores(max(ql-yt, yt-qu), scores)

        probs = weights / weights.sum()
        alpha_out = float(np.clip(np.dot(probs, expert_alpha), ALPHA_MIN, ALPHA_MAX))
        lo_t, hi_t = conformal_bounds(ql, qu, scores, alpha_out)
        err_t = interval_error(yt, lo_t, hi_t)

        # Expert losses for past expert states; update AFTER forecasting.
        expert_errors = np.array([
            interval_error(yt, *conformal_bounds(ql, qu, scores, a))
            for a in expert_alpha
        ], dtype=np.float64)
        losses = np.array([pinball_loss(beta, a, alpha) for a in expert_alpha])
        logw = np.log(weights) - eta*losses
        logw -= np.max(logw)
        wbar = np.exp(logw)
        wbar /= wbar.sum()
        weights = (1 - sigma)*wbar + sigma/k
        expert_alpha = np.clip(
            expert_alpha + gammas * (alpha - expert_errors),
            ALPHA_MIN, ALPHA_MAX
        )

        j = t-tinit
        alpha_traj[j], errors[j] = alpha_out, err_t
        lower[j], upper[j], truth[j] = lo_t, hi_t, yt
        if t % 200 == 0:
            print(f"[DtACI] t={t} alpha={alpha_out:.4f} err={err_t:.0f}")
    return alpha_traj, errors, upper, lower, truth


# ============================================================
# METHOD 5: AgACI
# ============================================================
class BOAEndpoint:
    """Adaptive Bernstein expert aggregation with pinball gradient trick.

    This is a transparent Python BOA-style implementation, not a verified
    numerical reproduction of the R OPERA package used by Zaffran et al.
    Separate instances are maintained for each endpoint.
    """
    def __init__(self, n_experts, tau, eps):
        self.k = n_experts
        self.tau = tau
        self.eps = eps
        self.p = np.full(n_experts, 1.0/n_experts)
        self.eta = np.zeros(n_experts)
        self.L = np.zeros(n_experts)
        self.squared = np.zeros(n_experts)
        self.max_abs = np.zeros(n_experts)

    def forecast(self, expert_values):
        return float(self.p @ expert_values)

    def update(self, expert_values, aggregated, observed):
        # d/d forecast of pinball loss rho_tau(y-forecast).
        gradient = float(observed < aggregated) - self.tau
        regrets = gradient * (expert_values - aggregated)
        self.squared += regrets**2
        self.max_abs = np.maximum(self.max_abs, np.abs(regrets))
        E = 2.0**(np.ceil(np.log2(self.max_abs + self.eps)) + 1.0)
        self.L += 0.5*(regrets*(1 + self.eta*regrets) +
                       E*(self.eta*regrets > 0.5))
        candidate = np.full(self.k, np.inf)
        active = self.squared > 0
        candidate[active] = np.sqrt(np.log(self.k)/self.squared[active])
        self.eta = np.minimum(1.0/E, candidate)
        logw = np.log(self.eta) - self.eta*self.L
        logw -= np.max(logw)
        w = np.exp(logw)
        self.p = w/w.sum()


def run_agaci(full_dataset, alpha=ALPHA, tinit=TINIT, splitSize=SPLIT,
              gamma_candidates=None, eps=AGACI_EPS):
    """Zaffran et al. ICML 2022 AgACI: separate endpoint BOA aggregation.

    At each t, every gamma expert creates an ACI interval; the lower and
    upper endpoint forecasts are aggregated with *separate* BOA weights.
    Infinite expert bounds are replaced by predefined finite training-only
    thresholds, as permitted by Algorithm 1's M^(l), M^(u).
    """
    if gamma_candidates is None:
        gamma_candidates = AGACI_GAMMA_CANDIDATES
    gammas = np.asarray(gamma_candidates, dtype=np.float64)
    k = len(gammas)
    if k < 2:
        raise ValueError("BOA aggregation needs at least two experts")
    set_seed(SEED)
    T = len(full_dataset)
    n_pred = T-tinit
    alpha_traj = np.full(n_pred, np.nan, np.float64)  # no single AgACI alpha!
    errors = np.empty(n_pred, np.float64)
    upper = np.empty(n_pred, np.float64)
    lower = np.empty(n_pred, np.float64)
    truth = np.empty(n_pred, np.float64)
    expert_alpha = np.full(k, alpha, np.float64)
    lower_boa = BOAEndpoint(k, alpha/2, eps)
    upper_boa = BOAEndpoint(k, 1-alpha/2, eps)
    quantiles = [alpha/2, 1-alpha/2]
    mlo, mhi, olo, ohi = _fresh_models()

    # Predeclare finite endpoints from TRAINING-ONLY raw-scale standardized
    # history. These are a stated experimental choice for M^(l), M^(u).
    initial = np.asarray(full_dataset.y[:tinit], dtype=float)
    yrange = float(np.ptp(initial))
    margin = 5.0 * max(yrange, 1e-8)
    m_lower = float(np.min(initial) - margin)
    m_upper = float(np.max(initial) + margin)

    for t in range(tinit, T):
        tr, cal = time_step_data(full_dataset, t, tinit, splitSize)
        _maybe_train(t, tinit, full_dataset.X[tr], full_dataset.y[tr],
                     mlo, mhi, olo, ohi, quantiles)
        cal_l, cal_u = predict_quantiles(mlo, mhi, full_dataset.X[cal], device)
        scores = np.maximum(cal_l-full_dataset.y[cal], full_dataset.y[cal]-cal_u)
        ql, qu = predict_quantiles(mlo, mhi, full_dataset.X[t:t+1], device)
        ql, qu = float(ql[0]), float(qu[0])
        yt = float(full_dataset.y[t])

        raw_intervals = [conformal_bounds(ql, qu, scores, a)
                         for a in expert_alpha]
        expert_errors = np.array([
            interval_error(yt, lo, hi) for lo, hi in raw_intervals
        ])
        # BOA requires bounded expert predictions. These thresholds apply
        # only to AgACI's endpoint experts, not to DtACI/LACI predictions.
        expert_lower = np.array([
            np.clip(lo, m_lower, m_upper) for lo, hi in raw_intervals
        ], dtype=float)
        expert_upper = np.array([
            np.clip(hi, m_lower, m_upper) for lo, hi in raw_intervals
        ], dtype=float)
        pred_lower = lower_boa.forecast(expert_lower)
        pred_upper = upper_boa.forecast(expert_upper)
        actual_err = interval_error(yt, pred_lower, pred_upper)

        # Observe target, update each BOA endpoint separately.
        lower_boa.update(expert_lower, pred_lower, yt)
        upper_boa.update(expert_upper, pred_upper, yt)
        # Each expert follows its own ACI feedback, not the aggregate error.
        expert_alpha = np.clip(
            expert_alpha + gammas * (alpha - expert_errors),
            ALPHA_MIN, ALPHA_MAX
        )

        j = t-tinit
        errors[j] = actual_err
        lower[j], upper[j], truth[j] = pred_lower, pred_upper, yt
        if t % 200 == 0:
            print(f"[AgACI] t={t} err={actual_err:.0f} "
                  f"interval=[{pred_lower:.3f}, {pred_upper:.3f}]")
    return alpha_traj, errors, upper, lower, truth


METHOD_FUNCS = {
   
    "ACI": run_aci,
    "LACI": run_laci,
    "DtACI": run_dtaci,
    "AgACI": run_agaci,
}


# ============================================================
# ALL-POINT EVALUATION (NO EXCLUDED INTERVALS OR LOCAL WINDOWS)
# ============================================================
def metrics_all_points(y_true, lower, upper, elapsed, alpha=ALPHA,
                       window=LOCAL_WINDOW):
    """Compute coverage and width metrics over every online forecast.

    A crossed interval is empty and thus contributes coverage 0 and width 0.
    An interval (-inf, inf) covers every finite target and has infinite width.
    Every overlapping full local window is retained.
    """
    y_true = np.asarray(y_true, dtype=np.float64)
    lower = np.asarray(lower, dtype=np.float64)
    upper = np.asarray(upper, dtype=np.float64)
    covered = (lower <= y_true) & (y_true <= upper)
    crossed = lower > upper
    width = np.maximum(upper - lower, 0.0)
    # np.maximum(-inf, 0) correctly gives zero width for empty prediction sets.
    n_windows = max(0, len(y_true) - window + 1)
    window_cov = np.array([
        float(np.mean(covered[i:i + window])) for i in range(n_windows)
    ], dtype=np.float64)
    window_width = np.array([
        float(np.mean(width[i:i + window])) for i in range(n_windows)
    ], dtype=np.float64)
    local_error = np.maximum((1.0 - alpha) - window_cov, 0.0)
    finite_cross = crossed & np.isfinite(lower) & np.isfinite(upper)
    return {
        "PICP": float(np.mean(covered)),
        "MPIW": float(np.mean(width)),
        "MLPICP": float(np.mean(window_cov)) if n_windows else np.nan,
        "LPICE": float(np.mean(local_error)) if n_windows else np.nan,
        "LPIA": float(np.mean(window_cov >= 1.0 - alpha)) if n_windows else np.nan,
        "LMPIW": float(np.mean(window_width)) if n_windows else np.nan,
        "LCWC": np.nan,  # External LCWC helper not specified for infinite/empty sets.
        "STD_LE": float(np.std(local_error)) if n_windows else np.nan,
        "n_local_windows": n_windows,
        "time_sec": float(elapsed),
        "n_finite_crossed": int(np.sum(finite_cross)),
        "n_excluded_windows": 0,
        "fraction_infinite": float(np.mean(~np.isfinite(lower) | ~np.isfinite(upper))),
    }


# ============================================================
# DRIVER
# ============================================================
def run_one_method(dataset_name, method_name, full_dataset, scaler, **kwargs):
    print(f"\n{'=' * 70}\n{dataset_name} :: {method_name}\n{'=' * 70}")
    run_fn = METHOD_FUNCS[method_name]
    start_time = time.time()
    alphaTraj, errSeq, y_u_preds, y_l_preds, y_true_vals = run_fn(full_dataset, **kwargs)
    time_taken = time.time() - start_time

    # Direct inverse affine mapping also preserves +/- infinity, which is
    # an intentional DtACI prediction-set boundary convention.
    offset = float(scaler.mean_[0])
    scale = float(scaler.scale_[0])
    y_u_inv = y_u_preds * scale + offset
    y_l_inv = y_l_preds * scale + offset
    y_true_inv = y_true_vals * scale + offset

    finite_cross = ((y_l_inv > y_u_inv) &
                    np.isfinite(y_l_inv) & np.isfinite(y_u_inv))
    all_finite = np.isfinite(y_u_inv).all() and np.isfinite(y_l_inv).all()
    if all_finite and not finite_cross.any():
        metrics = summarize_all_metrics(
            y_true_inv, y_l_inv, y_u_inv,
            alpha=ALPHA, w=LOCAL_WINDOW, eta=CWC_ETA,
            time_taken=time_taken,
        )
        metrics["n_finite_crossed"] = 0
        metrics["n_excluded_windows"] = 0
        metrics["fraction_infinite"] = 0.0
    else:
        metrics = metrics_all_points(y_true_inv, y_l_inv, y_u_inv, time_taken)
        print(f"[INFO] {method_name}: full evaluation retained every forecast and "
              f"every local window; crossed={metrics['n_finite_crossed']}, "
              f"unbounded_fraction={metrics['fraction_infinite']:.4f}. "
              "LCWC is NaN in this exceptional-interval branch.")

    print(f"[{dataset_name}/{method_name}] " +
          " | ".join(f"{k}={v:.4f}" for k, v in metrics.items()
                      if isinstance(v, float) and k != "time_sec"))
    print(f"[{dataset_name}/{method_name}] time_sec={time_taken:.2f}")

    if SAVE_TRAJECTORIES:
        os.makedirs(TRAJECTORY_DIR, exist_ok=True)
        max_len = max(len(alphaTraj), len(errSeq), len(y_u_inv), len(y_l_inv), len(y_true_inv))
        traj_df = pd.DataFrame({
            "step": np.arange(max_len),
            "alphaTrajectory": pad(alphaTraj, max_len),
            "errSeq": pad(errSeq, max_len),
            "y_u_pred": pad(y_u_inv, max_len),
            "y_l_pred": pad(y_l_inv, max_len),
            "y_true": pad(y_true_inv, max_len),
        })
        safe_ds = os.path.splitext(os.path.basename(dataset_name))[0]
        traj_df.to_csv(os.path.join(TRAJECTORY_DIR, f"{safe_ds}__{method_name}.csv"), index=False)

    return metrics


def main(dataset_paths):
    if not dataset_paths:
        print("No datasets provided. Edit DATASET_PATHS at the top of this "
              "script, or pass paths with --data path1.csv path2.csv ...")
        sys.exit(1)

    for path in dataset_paths:
        if not os.path.isfile(path):
            raise FileNotFoundError(f"Missing hydropower dataset: {path}")
    all_rows = []
    for dataset_path in dataset_paths:
        print(f"\n\n########## Loading dataset: {dataset_path} ##########")
        full_dataset, scaler, y_raw, y_scaled = load_dataset(dataset_path, seq_len=SEQ_LEN)
        print(f"Series length: {len(y_raw)}, usable (t - seq_len) pairs: {len(full_dataset)}")

        if len(full_dataset) <= TINIT:
            print(f"WARNING: dataset {dataset_path} has fewer than TINIT={TINIT} "
                  f"usable points ({len(full_dataset)}). Skipping.")
            continue

        for method_name in METHODS_TO_RUN:
            kwargs = dict(alpha=ALPHA, tinit=TINIT, splitSize=SPLIT)
            if method_name in ("ACI", "LACI"):
                kwargs["gamma"] = GAMMA
            elif method_name == "DtACI":
                kwargs["gamma_candidates"] = DTACI_GAMMA_CANDIDATES
                kwargs["interval_size"] = DTACI_INTERVAL_SIZE
            elif method_name == "AgACI":
                kwargs["gamma_candidates"] = AGACI_GAMMA_CANDIDATES
                kwargs["eps"] = AGACI_EPS

            metrics = run_one_method(dataset_path, method_name, full_dataset, scaler, **kwargs)
            row = {"dataset": os.path.basename(dataset_path), "method": method_name}
            row.update(metrics)
            all_rows.append(row)

            pd.DataFrame(all_rows).to_csv(OUTPUT_CSV, index=False)

    results_df = pd.DataFrame(all_rows)
    col_order = ["dataset", "method", "PICP", "MPIW", "MLPICP", "LPICE", "LPIA",
                 "LMPIW", "LCWC", "STD_LE", "n_local_windows", "time_sec"]
    col_order = [c for c in col_order if c in results_df.columns]
    results_df = results_df[col_order + [c for c in results_df.columns if c not in col_order]]
    results_df.to_csv(OUTPUT_CSV, index=False)

    print(f"\nAll results written to: {OUTPUT_CSV}")
    if SAVE_TRAJECTORIES:
        print(f"Per-step trajectories written to: {TRAJECTORY_DIR}/")
    print("\n", results_df.to_string(index=False))
    return results_df


if __name__ == "__main__":
    dataset_paths = DATASET_PATHS
    main(dataset_paths)

device: cuda


########## Loading dataset: ../Datasets/hydropower_dataset.csv ##########
hydropower_dataset.csv: source rows=1759, excluded missing/zero rows=6, retained=1753
Series length: 1753, usable (t - seq_len) pairs: 1657

../Datasets/hydropower_dataset.csv :: ACI
[ACI] t=200, alphat=0.1001, err_t=0
[ACI] t=400, alphat=0.1101, err_t=0
[ACI] t=600, alphat=0.1061, err_t=0
[ACI] t=800, alphat=0.1101, err_t=1
[ACI] t=1000, alphat=0.1041, err_t=0
[ACI] t=1200, alphat=0.0791, err_t=0
[ACI] t=1400, alphat=0.0661, err_t=0
[ACI] t=1600, alphat=0.0761, err_t=0
[WARN] ACI: excluded 1 finite crossed intervals and 24 full local windows. LCWC unavailable in this branch.
[../Datasets/hydropower_dataset.csv/ACI] PICP=0.8846 | MPIW=157.8958 | MLPICP=0.8854 | LPICE=0.0634 | LPIA=0.6901 | LMPIW=158.9479 | STD_LE=0.1416 | LCWC=nan | fraction_infinite=0.0000
[../Datasets/hydropower_dataset.csv/ACI] time_sec=7.74

../Datasets/hydropower_dataset.csv :: LACI
[LACI] t=200, alphat=0.1001, weighted_err=0.